<b><font size="6" color="#E8800A">Week 11a: Boosting for classification</font></b><br>
<b><font size="4">Training-only selection, one sealed test partition</font></b><br>

Boosting fits learners in sequence. The validation curves show how the base learner, stage count and learning rate change the result.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>AdaBoost</font>](#ada)
* [<font color='#E8800A'>AdaBoost settings</font>](#ada_settings)
* [<font color='#E8800A'>Gradient boosting</font>](#gb)
* [<font color='#E8800A'>Gradient boosting settings</font>](#gb_settings)
* [<font color='#E8800A'>Histogram gradient boosting</font>](#hist)
* [<font color='#E8800A'>Histogram boosting settings</font>](#hist_settings)
* [<font color='#E8800A'>Selection of the boosting model</font>](#sel1)
* [<font color='#E8800A'>The boosting winner</font>](#boost_winner)
* [<font color='#E8800A'>The boosting result</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

The test rows are sealed, the training folds are prepared once and the earlier learners are written out once, so every score in the notebook is read on the same rows.

In [ ]:
# The notebook folder holds preprocessing.py and course_helpers.py, so it joins the import path first.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Libraries for data handling, figures and parallel work.
import os
import json
import time
import warnings
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn: the models, the validation tools and the metrics used below.
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, cross_validate, train_test_split, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer
from sklearn.utils.validation import has_fit_parameter
# Progress bars, then the course helpers: the figure palette, the model log and the scoreboard.
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context
# The fold-owned preparation, stacking and blending classes written for this course.
from preprocessing import LockedMemory, weighted_ada_members, AdaRawFrame
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.patches import FancyBboxPatch

# One seed for every random choice, then the figure style: white grid, 150 dpi, vector output.
random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
set_matplotlib_formats("svg")
# The exact-message filter also reaches the parallel workers through the environment; the library message is expected, because a validation fold can hold a category the training rows lack.
os.environ["PYTHONWARNINGS"] = "ignore:Found unknown categories:UserWarning"
warnings.filterwarnings("ignore", message="Found unknown categories.*during transform", category=UserWarning)
print("scikit-learn", sklearn.__version__)
from preprocessing import save_boosting_selection

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


<div class="alert alert-block alert-info">
<code>AdaRawFrame()</code>

A transformer in `preprocessing.py`, beside this notebook.

**Definition:**
Makes missing values finite without learning statistics, returning raw rows as a NumPy array of Python objects with missing values replaced by a fixed text sentinel.

**Main Parameters:**
None.

**Main Methods:**
- `fit(X, y=None)`: returns the transformer unchanged.
- `transform(X)`: returns the array with missing values replaced.

**Differences from scikit-learn:**
[`FunctionTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html) applies a supplied function; `AdaRawFrame` fixes it. `SampleWeightedPreparedEstimator`, built by `weighted_ada_members`, restores missing values before applying the course recipe. AdaBoost can therefore take raw-row folds from `prepared_folds` without fitting preparation first.
</div>

__Step 1:__ Load the frame, seal the test partition, and prepare the training folds.

In [ ]:
# The champions table with the feature recipe of the feature-work notebook, indexed by athlete.
recipe_log = "../../logs/week_04_feature_work_classification_log.json"
frame, recipe = load_recipe("../../data/interim/champions.csv", recipe_log)
frame = frame.set_index("RecordID")

# Features and target, then the test rows are sealed once: 20 percent of the athletes, stratified by outcome, and no choice below reads them.
features = frame.loc[:, recipe["categorical"] + recipe["numeric"]].copy()
target = frame["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(features, target,
    test_size=0.2, random_state=random_state, stratify=target)
# `prepared`, `raw_prepared` and `ada_prepared` hold, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`): prepared with the recipe, left raw, and raw but made finite for the raw-row path of calibrated naive Bayes inside AdaBoost.
preparation = FoldPreprocessor(recipe)
folds = outer_splitter("classification", random_state=random_state)
prepared = prepared_folds(preparation, list(folds.split(X_train, y_train)), X_train, y_train)
raw_prepared = prepared_folds(FunctionTransformer(), list(folds.split(X_train, y_train)), X_train, y_train)
ada_prepared = prepared_folds(AdaRawFrame(), list(folds.split(X_train, y_train)), X_train, y_train)
# The recipe is fitted once on all the training rows. Every model fitted on all the rows learns from `Z_train` and is scored on `Z_test`, the same prepared rows, so no model refits the recipe.
full_prep = clone(preparation).fit(X_train, y_train)
Z_train = full_prep.transform(X_train)
Z_test = full_prep.transform(X_test)
# `model_log` collects every logged result in order; `results` is the folder of saved results and of this notebook's fitted winner.
model_log = []
results_dir = "../../results"
print("training folds:", folds.get_n_splits())

The test partition stays fixed. Validation alone chooses candidates; test columns are descriptive checkpoints. Prepared matrices belong to their training folds. Probability calibration and the downstream stacking and blending notebook need additional internal splits, so members receive raw rows and fit preparation after those splits.

__Step 2:__ Write the earlier learners once: the stored winner of each earlier algorithm and the best evaluated bagging and forest.

In [ ]:
taught_learners = {
    "logistic regression": LogisticRegression(solver="saga", l1_ratio=1.0, C=100, fit_intercept=False, max_iter=5000, random_state=random_state),
    "naive Bayes": CalibratedClassifierCV(GaussianNB(var_smoothing=0.1, priors=None), method="isotonic"),
    "KNN": KNeighborsClassifier(n_neighbors=41, p=1, weights="distance"),
    "pruned tree": DecisionTreeClassifier(ccp_alpha=0.001, class_weight=None, criterion="gini", max_depth=None, max_features=None, min_samples_leaf=20, min_samples_split=50, splitter="best", random_state=random_state),
    "bagged trees": BaggingClassifier(estimator=DecisionTreeClassifier(random_state=33), n_estimators=139, n_jobs=4, random_state=33),
    "random forest": RandomForestClassifier(n_estimators=127, n_jobs=4, random_state=33),
}

<div class="alert alert-block alert-info">
<code>weighted_ada_members(preprocessor, estimators, memory=None)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:**
Wraps each eligible earlier learner in a `SampleWeightedPreparedEstimator`, a `PreparedEstimator` subclass whose `fit` accepts `sample_weight`, so AdaBoost can use raw rows.

**Main Parameters:**
- `preprocessor`: unfitted recipe, cloned into every member.
- `estimators`: dictionary from name to learner.
- `memory`: optional joblib `Memory` caching the preprocessing fit.

**Returns:**
A dictionary with the same names. Each `CalibratedClassifierCV` becomes a member with its base learner and `calibration` set to its method; other learners are wrapped unchanged.

**Differences from scikit-learn:**
[`AdaBoostClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) passes `sample_weight` to its base learner's `fit`. The wrapper forwards it to the learner, including `CalibratedClassifierCV.fit` when calibrated. Calibration splits raw rows before preparing them inside each fold; a plain base learner receives rows prepared beforehand.
</div>

__Step 3:__ Print the stored labels beside the constructors and identify AdaBoost eligibility.

<div class="alert alert-block alert-info">
<code>LockedMemory(location=None, backend="local", mmap_mode=None, compress=False, verbose=1, backend_options=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
A joblib `Memory` with a lock file keyed by the function and its arguments. A second worker waits for the first computation, then reads its cache entry.

**Main Parameters:**
- `location`: cache folder; `None` switches caching off.
- Other parameters: those of joblib `Memory`.

**Main Methods:**
- `cache(func)`: the cached, locked version of `func`.

**Differences from scikit-learn:**
scikit-learn uses joblib [`Memory`](https://joblib.readthedocs.io/en/latest/user_guide/memory.html), for example in `Pipeline(memory=...)`, which allows concurrent workers to compute the same missing entry. `LockedMemory` prevents repeated preprocessing fits when the `filelock` package is installed. Without it, locking is skipped and duplicate computations give the same result.
</div>

In [ ]:
# Print each stored winner beside the constructor written above, so the two can be compared.
for (name, learner), week in zip(list(taught_learners.items())[:4], range(6, 10)):
    stored = json.loads(Path(results_dir, f"week_{week:02d}.json").read_text(encoding="utf-8"))["classification"]["model"]
    print(f"{name} | week_{week:02d}: {stored}")
    print("constructor:", learner)
# The bagged trees and the forest have no stored label: they are the best tested configurations of the previous notebook.
for name, learner in list(taught_learners.items())[4:]:
    print(f"{name} | best evaluated in the previous week")
    print("constructor:", learner)
# AdaBoost refits its base on reweighted rows, so every learner is checked for a `sample_weight` argument in `fit`.
sample_weight_support = {name: has_fit_parameter(learner, "sample_weight") for name, learner in taught_learners.items()}
# Only learners whose `fit` accepts `sample_weight` can serve as an AdaBoost base: ensembles shrink to 16 members because every stage refits the base, and logistic regression gets 20000 iterations because reweighted rows converge more slowly.
eligible_ada = {name: clone(learner).set_params(**{key: value for key, value in dict(max_samples=None, n_estimators=16, max_iter=20000).items() if key in learner.get_params(deep=False)}) for name, learner in taught_learners.items() if sample_weight_support[name]}
# `ada_members` wraps each eligible learner with its own raw-row preparation; the cache folder keeps shared preparation from being repeated.
ada_members = dict(eligible_ada)
ada_members.update(weighted_ada_members(preparation, {name: learner for name, learner in eligible_ada.items() if isinstance(learner, CalibratedClassifierCV)}, memory=LockedMemory(".w9_preprocessing_cache", verbose=0)))
# Say which learners can serve as a base and which cannot.
print("base fit sample_weight signatures:", sample_weight_support)
print("AdaBoost eligible:", list(eligible_ada))
print("AdaBoost excluded:", [name for name in taught_learners if name not in eligible_ada])
# `member_memory` stores the predictions of each combiner member for identical fit and score rows, so a combination that reuses a member never refits it; it is a `LockedMemory`, so workers that need the same entry wait for one computation.
member_memory = LockedMemory(".ensemble_member_cache", verbose=0)

AdaBoost requires a base estimator whose `fit` accepts `sample_weight`: logistic regression, calibrated Gaussian naive Bayes, the pruned tree, bagged trees and the forest qualify; KNN does not. Boosting weights sum to one, so a float `max_samples` such as 1.0 would mean one row per bootstrap sample (scikit-learn warns). Bagged trees and the forest therefore use `max_samples=None`, drawing as many rows as the learner holds. They use 16 members instead of 139 and 127 because every boosting stage refits them. Logistic regression uses 20000 iterations instead of 5000 because reweighted rows converge more slowly.

All taught learners remain in voting, stacking and blending. Preparation is fitted once per fold and once on all training rows, except for calibrated naive Bayes: it takes raw rows and prepares inside its own calibration splits. [AdaBoostClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)

# <font color='#E8800A'>AdaBoost</font> <a class="anchor" id="ada"></a>
[Back to TOC](#toc)

The next learner receives a changed view of the errors. AdaBoost classification increases the relative weight of misclassified rows; AdaBoost.R2 uses a normalised prediction loss and a weighted median of member predictions.

__Step 4:__ Fit a stump to six toy rows and compute the first weight update by hand.

In [ ]:
# Six rows with one input and a binary class, all starting with the same weight.
toy_x = np.arange(6).reshape(-1, 1)
toy_y = np.array([0, 0, 1, 0, 1, 1])
toy_weights = np.full(6, 1 / 6)
# One split is fitted with the row weights; the weighted error of its wrong rows gives the stage weight.
toy_stump = DecisionTreeClassifier(max_depth=1, random_state=random_state).fit(toy_x, toy_y, sample_weight=toy_weights)
toy_wrong = toy_stump.predict(toy_x) != toy_y
toy_error = np.dot(toy_weights, toy_wrong)
toy_alpha = np.log((1 - toy_error) / toy_error)
# Wrong rows are multiplied up and the weights are normalised to sum to one; the table sets old and next weights side by side.
toy_next = toy_weights * np.exp(toy_alpha * toy_wrong)
toy_next /= toy_next.sum()
print(pd.DataFrame({"class": toy_y, "prediction": toy_stump.predict(toy_x), "wrong": toy_wrong, "old weight": toy_weights, "next weight": toy_next}).round(4).to_string(index=False))
print("weighted error:", toy_error, "alpha:", toy_alpha)

For [binary SAMME](https://doi.org/10.4310/SII.2009.v2.n3.a8) with learning rate one, $E=\sum_i w_i\mathbf{1}[y_i\ne h(x_i)]$ and $\alpha=\log((1-E)/E)$. Here $i$ indexes rows, $w_i$ is the current normalised weight, $y_i$ is its label, $x_i$ is its input, $h$ is the fitted stump and the indicator is one for a wrong answer. Multiply wrong rows by $\exp(\alpha)$, leave correct rows unchanged, then normalise all weights to sum to one.

__Step 5:__ Draw the fitted toy stump, including every node.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plot_tree(toy_stump, feature_names=["toy position"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("AdaBoost: the first toy learner")
plt.tight_layout()
plt.show()

**The stump misclassifies one of the six rows, so the weighted error is 0.1667 and alpha is 1.6094.** The misclassified row's weight rises from 0.1667 to 0.5, half of the total, while each correct row falls to 0.1, so the next learner is fitted mainly to that row.

__Step 6:__ Draw three stages of AdaBoost on the six toy rows: marker area is the row weight, orange marks a row that the stump of the stage gets wrong, and each title gives the stage weight, the size of that stump's vote in the final answer.

In [ ]:
# Start from equal row weights; `toy_x` and `toy_y` are the six toy rows of the step above.
stage_weights = np.full(6, 1 / 6)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
# One panel per stage: fit a stump with the current weights, then draw rows sized by weight (orange where the stump is wrong) and the stump boundary.
for stage, axis in tqdm(enumerate(axes, start=1), desc='stage, axis', leave=True, position=0):
    stump = DecisionTreeClassifier(max_depth=1, random_state=random_state).fit(toy_x, toy_y, sample_weight=stage_weights)
    wrong = stump.predict(toy_x) != toy_y
    error = np.dot(stage_weights, wrong)
    alpha = np.log((1 - error) / error)
    axis.scatter(toy_x[:, 0], toy_y, s=3000 * stage_weights, c=np.where(wrong, PLOT_ORANGE, PLOT_BLUE), edgecolor="white", zorder=3)
    axis.axvline(stump.tree_.threshold[0], color="black", linestyle="--")
    axis.set(title=f"stage {stage}: stage weight {alpha:.2f}", xlabel="toy position", yticks=[0, 1], ylim=(-0.5, 1.5))
    print(f"stage {stage}: weighted error {error:.4f}, stage weight {alpha:.4f}, row weights {stage_weights.round(3)}")
    # Raise the weight of the wrong rows for the next stump and keep the weights summing to one.
    stage_weights = stage_weights * np.exp(alpha * wrong)
    stage_weights /= stage_weights.sum()
axes[0].set_ylabel("class")
plt.tight_layout()
plt.show()

**The stage weight, the size of a stump's vote, is 1.6094 at stage 1, 2.1972 at stage 2 and 1.2528 at stage 3, with weighted errors of 0.1667, 0.1000 and 0.2222.** Marker area shows each row's weight: after stage 1 the one wrong row, drawn orange, holds half of the total weight (0.5 in the printed weights), so the stump of stage 2 is fitted mainly to it.

__Step 7:__ Draw one AdaBoost stage as a flow: what goes in, what is fitted and what is passed on.

In [ ]:
# Each box is a patch on a blank axis: what goes into one AdaBoost stage, what is fitted and what is passed on. Arrows show the order inside one stage.
stage_boxes = ['row weights w\n(they sum to one)', 'fit a learner h\non the weighted rows', 'weighted error E\nand stage weight alpha', 'raise the weights of\nthe wrong rows: new w']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor=PLOT_BLUE, linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the new weights are the input of the next stage", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**One AdaBoost stage takes the row weights in and passes updated weights and a stage weight on.** Follow the arrows: the learner is fitted to the weighted rows, its weighted error sets its stage weight, and the wrong rows gain weight before the next stage.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html'>sklearn.ensemble.AdaBoostClassifier(estimator=None, n_estimators=50, learning_rate=1.0, random_state=None)</a>

**Definition:**
Fits a sequence of base learners on reweighted rows, so that each stage concentrates on the rows the earlier stages misclassified, and combines them by a weighted vote.

**Main Parameters:**
- `estimator`: base learner; it must accept `sample_weight`.
- `n_estimators`: maximum number of stages.
- `learning_rate`: multiplier on the stage weights.
- `random_state`: seed of the base learner when it uses one.

**Main Attributes (after fitting):**
- `estimators_`: the fitted stages.
- `estimator_weights_`: the weight of each stage in the vote.
- `estimator_errors_`: the weighted error of each stage.

**Main Methods:**
- `fit(X, y, sample_weight=None)`: fit the stages in sequence.
- `predict(X)`: the weighted vote.
- `staged_predict(X)`: the prediction after each stage.
</div>

The reference uses the pruned tree as its base learner; the sweeps below change one setting of that reference, and the estimator sweep compares every eligible base learner.

__Step 8:__ Score the AdaBoostClassifier reference on the ten prepared training folds.

In [ ]:
ada_reference = AdaBoostClassifier(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
ada_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
ada_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 9:__ Fit the AdaBoostClassifier reference on all the training rows.

In [ ]:
ada_default = clone(ada_reference).fit(Z_train, y_train)
ada_default_configuration = f"AdaBoostClassifier(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in ada_reference.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostClassifier().get_params(deep=False)[setting_name]]) + ")"
ada_complete_rows = []

__Step 10:__ Log the AdaBoostClassifier reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_default_configuration, "train": ada_default_train, "validation": ada_default_validation,
    "model": ada_default},
    Z_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>AdaBoost settings</font> <a class="anchor" id="ada_settings"></a>
[Back to TOC](#toc)

Each sweep changes one AdaBoost setting from the reference and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; interactions need the later joint search.

| Parameter | Meaning |
| --- | --- |
| `estimator` | Base learner refitted at every stage. |
| `n_estimators` | Maximum number of stages. |
| `learning_rate` | Multiplier on each stage's weight and on its reweighting of rows. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)

__Step 11:__ Change only AdaBoost `estimator` on the fixed training folds, using the eligible learners of `ada_members`.

### `AdaBoost: estimator`

Each stage refits a learner on reweighted rows, so its `fit` must accept `sample_weight`. The default `None` builds a depth-one decision stump. Stronger learners are expected to leave fewer errors to correct and cost more per stage. The candidates are the eligible taught base estimators with their own winning settings fixed, and KNN is absent because it does not accept `sample_weight`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
ada_estimator_values = list(ada_members.values())
# One row per value will hold its scores, its seconds and its fitted model.
ada_estimator_rows = []
for name, value in tqdm(list(zip(ada_members, ada_estimator_values)), desc="AdaBoostClassifier.estimator", unit="value"):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(estimator=value)
    # Calibrated naive Bayes is the one wrapped base learner: it takes raw rows and prepares inside its own calibration splits. Every other base learner takes rows the recipe already prepared.
    raw_path = isinstance(value, PreparedEstimator)
    folds_used = ada_prepared if raw_path else prepared
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in folds_used]
        + [joblib.delayed((PreparedEstimator(AdaRawFrame(), clone(candidate)) if raw_path else clone(candidate)).fit)((X_train if raw_path else Z_train), y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(folds_used, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(folds_used, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    ada_estimator_rows.append({"configuration": f"AdaBoostClassifier(" + ", ".join([f"estimator={name}"] + [f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostClassifier().get_params(deep=False)[setting_name]]) + ")", "setting": name, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict((X_test if raw_path else Z_test))), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
ada_estimator_sweep = pd.DataFrame(ada_estimator_rows)
ada_complete_rows.extend(ada_estimator_rows)
ada_estimator_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_estimator_winner = ada_estimator_sweep.loc[ada_estimator_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_estimator_winner["configuration"], "train": ada_estimator_winner["train"],
    "validation": ada_estimator_winner["validation"], "model": ada_estimator_winner["model"]},
    X_test if isinstance(ada_estimator_winner["model"], PreparedEstimator) else Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Changing only the base learner gives the best validation F1 to random forest: 0.8613 against 0.8428 for the default stump.** The stage and learning-rate sweeps below return to that stump.

AdaBoost reweights earlier mistakes, so a new stage contributes where its base learner still errs on the weighted training rows. For standard binary AdaBoost with learning rate one, let $\varepsilon_t$ be the weighted training error at stage $t$ and $\gamma_t=\tfrac{1}{2}-\varepsilon_t$ its edge over guessing. After $T$ stages the combined training error is at most $\prod_{t=1}^{T} 2\sqrt{\varepsilon_t(1-\varepsilon_t)}\le\exp\left(-2\sum_{t=1}^{T}\gamma_t^2\right)$ ([Freund and Schapire, 1997](https://doi.org/10.1006/jcss.1997.1504)). Here $t$ numbers stages, $T$ is their count, the product has one factor per stage, the sum adds their squared edges and $\exp$ is the exponential function. A positive edge lowers the bound; a larger edge lowers it more.

A weak learner is only somewhat better than chance, and successive stages can help if they retain a positive edge after reweighting. An elaborate learner may fit the weighted rows almost perfectly immediately, giving its first stage a very large weight and leaving little for later stages; scikit-learn stops when $\varepsilon_t=0$. Or it may refit nearly the same function, adding cost without new edge.

Unstable does not mean weak. An unpruned deep tree is unstable but fits its training rows perfectly, leaving nothing to reweight. Bagging helps by averaging away its variation; boosting does not.

Sources: [Freund and Schapire, 1997](https://doi.org/10.1006/jcss.1997.1504) and the [scikit-learn AdaBoost guide](https://scikit-learn.org/stable/modules/ensemble.html#adaboost).

__Step 12:__ Fit each base learner alone and boosted with the AdaBoost defaults on the ten prepared folds, and score every stage.

In [ ]:
# From the weakest learner to the most elaborate: the shallow trees, the unpruned tree and the learners of the estimator sweep.
comparison_learners = {
    "default stump": DecisionTreeClassifier(max_depth=1, random_state=random_state),
    "deep tree": DecisionTreeClassifier(random_state=random_state),
    **ada_members}
# The AdaBoost defaults, 50 stages at a learning rate of 1.0; only the base learner changes.
default_boost = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=random_state)
comparison_rows, staged_curves = [], {}
for name, learner in tqdm(comparison_learners.items(), desc="base learners", unit="learner", leave=True, position=0):
    # Calibrated naive Bayes takes the raw rows of its own fold preparation; every other learner takes rows the recipe already prepared.
    folds_used = ada_prepared if isinstance(learner, PreparedEstimator) else prepared
    # Alone: the ten fold fits run at the same time and are scored one by one afterwards.
    started = time.perf_counter()
    alone_fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(learner).fit)(fold["X_fit"], fold["y_fit"]) for fold in folds_used)
    alone_scores = np.array([f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(folds_used, alone_fits)])
    alone_seconds = time.perf_counter() - started
    # Boosted: the same folds, with the learner as the base of AdaBoost.
    started = time.perf_counter()
    boosted_fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(default_boost).set_params(estimator=learner).fit)(fold["X_fit"], fold["y_fit"]) for fold in folds_used)
    boosted_seconds = time.perf_counter() - started
    # Keep actual fitted stages; a curve stops when the shortest fold stops.
    staged = []
    for fold, fitted in zip(folds_used, boosted_fits):
        answers = list(fitted.staged_predict(fold["X_score"]))
        staged.append([f1_score(fold["y_score"], answer) for answer in answers])
    boosted_scores = np.array([curve[-1] for curve in staged])
    common_stages = min(map(len, staged))
    staged_curves[name] = np.mean([curve[:common_stages] for curve in staged], axis=0)
    # One row per learner: the scores, the gain, the spread over the folds, the stages kept, the error of the first stage and the seconds.
    comparison_rows.append({"learner": name, "alone F1": alone_scores.mean(), "boosted F1": boosted_scores.mean(),
        "gain": boosted_scores.mean() - alone_scores.mean(),
        "SD alone": alone_scores.std(ddof=1), "SD boosted": boosted_scores.std(ddof=1),
        "stages kept": np.mean([len(fitted.estimators_) for fitted in boosted_fits]),
        "first-stage error": np.mean([fitted.estimator_errors_[0] for fitted in boosted_fits]),
        "seconds alone": alone_seconds, "seconds boosted": boosted_seconds})
base_comparison = pd.DataFrame(comparison_rows).set_index("learner")

__Step 13:__ Print the scores, the gain, the spread, the stages kept, the first-stage error and the seconds of every base learner.

In [ ]:
# The scores alone and boosted, the gain, the spread over the ten folds, the stages kept, the weighted error of the first stage and the seconds.
display(base_comparison.round(4))

Gain is the increase in F1 over the learner alone. Standard deviations describe variation across the ten fold scores in the same units; they are not prediction errors. Stages kept averages `len(estimators_)`, the stages left when boosting ends. First-stage error averages `estimator_errors_[0]`, the first stage's weighted error. Seconds measure wall time for the ten fold fits at four workers.

**Boosting lifts the default stump from F1 0.7923 to 0.8428, a gain of 0.0505 over fifty stages, and no other learner gains as much.** Naive Bayes gains 0.0261 and keeps 39.3 stages on average, the forest gains 0.0068 and bagged trees 0.0045, while the pruned tree loses 0.0034 and logistic regression loses 0.0262 with 7.1 stages kept. The deep tree gains 0.0000: its first-stage weighted error is 0.0000, and boosting keeps one stage. Bagged trees and the forest start with first-stage weighted errors of 0.0054 and 0.0034, against 0.2556 for the stump. The fold standard deviation falls from 0.0183 to 0.0167 for the stump, from 0.0161 to 0.0111 for bagged trees and from 0.0182 to 0.0119 for the pruned tree. 

__Step 14:__ Plot the score after every stage and the gain of every base learner.

In [ ]:
# Each curve starts at the first fitted stage and ends when the shortest fold stops.
fig, ax = plt.subplots(figsize=(9, 5))
colours = sns.color_palette("colorblind", len(staged_curves))
for colour, (name, curve) in zip(colours, staged_curves.items()):
    ax.plot(np.arange(1, len(curve) + 1), curve, color=colour, linewidth=2, label=name)
    ax.scatter([len(curve)], [curve[-1]], color=colour, marker="x" if len(curve) < 50 else "o", zorder=3)
ax.legend(fontsize=8)
ax.set(xlabel="actual boosting stages", ylabel="Validation F1", title="AdaBoost: validation after each fitted stage")
plt.tight_layout()
plt.show()

**In the left panel each line starts from the score of its learner alone and ends at its boosted score after fifty stages: the stump rises from 0.7923 to 0.8428 and naive Bayes from 0.8075 to 0.8336, while logistic regression falls from 0.8511 to 0.8249.** The deep tree is one point, 0.7996, because boosting stops after the first stage. In the right panel the bars are the gains of the table above, 0.0505 for the stump against 0.0068 for the forest.

__Step 15:__ Change only AdaBoost `n_estimators` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `AdaBoost: n_estimators`

Each stage fits another base learner, up to this limit; the library default is `50`. More stages can correct remaining errors, but validation may flatten or worsen as the model follows noise. Fitting stops early if the update cannot continue. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
ada_n_estimators_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
ada_n_estimators_rows = []
ada_stage_count = max(ada_n_estimators_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
ada_train_by_fold = np.zeros((len(prepared), ada_stage_count))
ada_validation_by_fold = np.zeros((len(prepared), ada_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).set_params(n_estimators=ada_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="AdaBoostClassifier staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(ada_stage_count):
        used = min(stage, len(on_fit) - 1)
        ada_train_by_fold[position, stage] = f1_score(fold["y_fit"], on_fit[used])
        ada_validation_by_fold[position, stage] = f1_score(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(ada_n_estimators_values[:-1], desc='ada n estimators values[:-1]', leave=True, position=0):
    fresh = clone(ada_reference).set_params(n_estimators=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.array_equal(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", ada_n_estimators_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(ada_n_estimators_values, desc='AdaBoostClassifier.n_estimators', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(n_estimators=value)
    train_scores = list(ada_train_by_fold[:, value - 1])
    validation_scores = list(ada_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    ada_n_estimators_rows.append({"configuration": f"AdaBoostClassifier(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostClassifier().get_params(deep=False)[setting_name]]) + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
ada_n_estimators_sweep = pd.DataFrame(ada_n_estimators_rows)
ada_complete_rows.extend(ada_n_estimators_rows)
ada_n_estimators_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_n_estimators_winner = ada_n_estimators_sweep.loc[ada_n_estimators_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_n_estimators_winner["configuration"], "train": ada_n_estimators_winner["train"],
    "validation": ada_n_estimators_winner["validation"], "model": ada_n_estimators_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The default stump scores validation F1 0.8428 at 50 stages; the best tested count is 130, scoring 0.8491.** Its training F1 is 0.8599.

__Step 16:__ Plot training and validation F1 against the number of AdaBoost stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
ada_stages = np.arange(1, ada_stage_count + 1)
ax.plot(ada_stages, ada_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(ada_stages, ada_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(ada_n_estimators_sweep["setting"], ada_n_estimators_sweep["train_F1"], color=PLOT_ORANGE, zorder=3)
ax.scatter(ada_n_estimators_sweep["setting"], ada_n_estimators_sweep["cv_F1"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="n_estimators", ylabel="F1", title="AdaBoostClassifier: train against validation", ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.show()

__Step 17:__ Change only AdaBoost `learning_rate` on the fixed training folds, with the sweep list `[0.1, 0.5, 1.0]`.

### `AdaBoost: learning_rate`

This rate scales the stage weight and the update of row weights; the default is `1.0`. A smaller rate weakens each correction, so it is expected to need more stages. Validation determines whether the slower sequence helps. The candidate list is `[0.1, 0.5, 1.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
ada_learning_rate_values = [0.1, 0.5, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
ada_learning_rate_rows = []
for value in tqdm(ada_learning_rate_values, desc="AdaBoostClassifier.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    ada_learning_rate_rows.append({"configuration": f"AdaBoostClassifier(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostClassifier().get_params(deep=False)[setting_name]]) + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
ada_learning_rate_sweep = pd.DataFrame(ada_learning_rate_rows)
ada_complete_rows.extend(ada_learning_rate_rows)
ada_learning_rate_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_learning_rate_winner = ada_learning_rate_sweep.loc[ada_learning_rate_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_learning_rate_winner["configuration"], "train": ada_learning_rate_winner["train"],
    "validation": ada_learning_rate_winner["validation"], "model": ada_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**With the default stump and 50 stages fixed, validation F1 is 0.8111 at rate 0.1 and 0.8428 at rate 1.0.** Shrinking each correction without adding stages hurts here.

__Step 18:__ Plot training and validation F1 for every other swept AdaBoost setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
ada_settings_tables = {"learning_rate": (ada_learning_rate_sweep, ada_learning_rate_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(1, 1, figsize=(4, 3.6), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), ada_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="F1", ylim=(0, 1.05))
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(ada_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**The rate sweep keeps the default base and 50 stages fixed: validation F1 changes from 0.8111 at 0.1 to 0.8428 at 1.0.** Read this as a one-setting comparison, separate from the rate-by-stage study.

__Step 19:__ Select the best tested AdaBoost configuration from the existing sweep rows by validation score.

A smaller learning rate makes each correction smaller, so it often needs more stages. These curves vary the rate and stage budget together while keeping the default base learner fixed. Validation decides whether extra stages help: noise, overfitting and early termination can break the pattern. Equal rate-times-stage products need not produce equal models.

__Step 20:__ Read the validation score after each actual stage at each learning rate.

In [ ]:
# The default base is fixed; this separate interaction varies the rate and stage budget.
ada_curve_rates = ada_learning_rate_values
ada_rate_curves = {}
for rate in tqdm(ada_curve_rates, desc="AdaBoost learning-rate interaction", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).set_params(learning_rate=rate, n_estimators=150).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    curves = [[f1_score(fold["y_score"], answer) for answer in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)]
    actual_stages = min(map(len, curves))
    ada_rate_curves[rate] = np.mean([curve[:actual_stages] for curve in curves], axis=0)
fig, ax = plt.subplots(figsize=(8, 5))
for rate, curve in ada_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter([len(curve)], [curve[-1]], marker="x" if len(curve) < 150 else "o", zorder=3)
ax.set(xlabel="actual boosting stages", ylabel="validation F1", title="AdaBoost: rate and fitted stages")
ax.legend()
plt.tight_layout()
plt.show()
print("Each curve ends when its shortest fold stops; x marks early termination.")
print(pd.DataFrame([{"learning_rate": rate, "actual stages": len(curve), "best stage": int(np.argmax(curve)) + 1, "best validation F1": curve.max()} for rate, curve in ada_rate_curves.items()]).round(4).to_string(index=False))

**Within the 150-stage budget, rate 0.1 peaks at validation F1 0.8261, while rate 1.0 reaches 0.8500.** More stages do not make the rates interchangeable.

__Step 21:__ Select the best complete AdaBoost configuration actually evaluated.

In [ ]:
ada_complete = pd.DataFrame(ada_complete_rows)
ada_best = ada_complete.loc[ada_complete["cv_F1"].idxmax()]
ada_best_parameters = ada_best["parameters"]
ada_best_configuration = clone(ada_reference).set_params(**ada_best_parameters)
print("best tested AdaBoost configuration:", ada_best["configuration"])

**The best complete AdaBoost configuration evaluated in the isolated sweeps uses random forest, with validation F1 0.8613.** The joint search later tests parameter combinations rather than combining separate sweep winners by assumption.

# <font color='#E8800A'>Gradient boosting</font> <a class="anchor" id="gb"></a>
[Back to TOC](#toc)

Gradient boosting also adds learners in sequence, but it does not reweight rows. Each new regression tree is fitted to the negative gradient of the loss at the predictions of the earlier trees, called the pseudo-residual. For squared error this is the residual, so a small price example makes the correction visible. [Gradient boosting guide](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting)

Both methods build the model one learner at a time. They differ in what a stage sees and in how the stages are added up.

| | AdaBoost | Gradient boosting |
|---|---|---|
| What a stage fits | A copy of the base learner, refitted on rows reweighted so that earlier mistakes count more | A regression tree fitted to the pseudo-residuals, loss-specific negative gradients or subgradients at the current predictions |
| Loss | Forward stagewise fitting of the exponential loss ([Friedman, Hastie and Tibshirani, 2000](https://doi.org/10.1214/aos/1016218223)) | Supported losses: `log_loss` and `exponential` for classification, `squared_error`, `absolute_error`, `huber` and `quantile` for regression |
| Base learner | Any learner that accepts `sample_weight`, as Step 3 shows | A regression tree, also for classification, because the pseudo-residuals are real numbers |
| How stages combine | Each stage receives its own weight from its weighted error, and the prediction is a weighted vote | The trees add up to one raw score, each scaled by the same learning rate; for binary `log_loss` the raw score is the log-odds; `exponential` uses half that scale |
| What a row hands to the next stage | A larger weight if it was misclassified | Its loss-specific pseudo-residual; under `log_loss`, the label minus the current predicted probability |

AdaBoost chooses stage weights and weighted base learners that minimise the exponential loss in forward stagewise fitting, producing its reweighting rule. Gradient boosting keeps stagewise fitting but allows other losses: a regression tree fits the direction in which loss falls fastest. For binary `log_loss`, this pseudo-residual is $r_i=y_i-p_i$: $i$ indexes rows, $y_i$ is a label coded 0 or 1, and $p_i$ is the ensemble's current probability, the sigmoid of its raw score. The tree's real-valued leaves move that score rather than predict class labels. With `loss='exponential'`, scikit-learn documents that gradient boosting recovers AdaBoost, which is why the sweep includes it.

__Step 22:__ Draw three stages of gradient boosting on twelve toy rows: the prediction after the earlier trees, the residuals and the tree fitted to them.

In [ ]:
# Twelve rows of one smooth curve; `grid` is a fine set of positions for drawing each tree as a step function.
sketch_x = np.linspace(0, 1, 12).reshape(-1, 1)
sketch_y = 2 + np.sin(5 * sketch_x[:, 0])
grid = np.linspace(0, 1, 200).reshape(-1, 1)
# Stage zero predicts the mean everywhere; each stage then adds a small tree fitted to what is left over.
prediction = np.full(12, sketch_y.mean())
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
# Top row: the prediction after the earlier trees (the mean at the start) with an arrow from it to each response (the residual). Bottom row: the tree fitted to those residuals.
for stage in tqdm(range(3), desc='stage', leave=True, position=0):
    residual = sketch_y - prediction
    tree = DecisionTreeRegressor(max_depth=2, random_state=random_state).fit(sketch_x, residual)
    axes[0, stage].scatter(sketch_x[:, 0], sketch_y, color=PLOT_BLUE, zorder=3)
    axes[0, stage].step(sketch_x[:, 0], prediction, where="mid", color=PLOT_ORANGE, linewidth=2)
    axes[0, stage].quiver(sketch_x[:, 0], prediction, np.zeros(12), residual, angles="xy", scale_units="xy", scale=1, width=0.004, color="gray")
    axes[0, stage].set_title("starting prediction: the mean" if stage == 0 else f"prediction after {stage} tree" + ("s" if stage > 1 else "") + ", with residuals")
    axes[1, stage].scatter(sketch_x[:, 0], residual, color=PLOT_BLUE, zorder=3)
    axes[1, stage].plot(grid[:, 0], tree.predict(grid), color=PLOT_ORANGE, linewidth=2, drawstyle="steps-mid")
    axes[1, stage].set_title(f"tree {stage + 1}, fitted to the residuals above")
    print(f"stage {stage + 1}: mean absolute residual {np.abs(residual).mean():.4f}")
    # The tree's answer is added after shrinking it by the learning rate, here 0.6.
    prediction = prediction + 0.6 * tree.predict(sketch_x)
axes[0, 0].set_ylabel("response")
axes[1, 0].set_ylabel("residual")
plt.tight_layout()
plt.show()

**The mean absolute residual of the twelve toy rows falls from 0.6346 before the first tree to 0.3168 after it and 0.2062 after two trees.** In the top row each grey arrow runs from the prediction after the earlier trees to a row's response; the tree below it is fitted to exactly those arrow lengths.

__Step 23:__ Draw one gradient boosting stage as a flow: what goes in, what is fitted and what is passed on.

In [ ]:
# Each box is a patch on a blank axis: what goes into one gradient boosting stage, what is fitted and what is passed on. Arrows show the order inside one stage.
stage_boxes = ['prediction F after the earlier trees\n(the mean at the start)', 'pseudo-residuals r:\nthe negative gradient of the loss', 'fit a regression tree h\nto the pseudo-residuals', 'F + learning rate x h:\na shrunk update']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor=PLOT_ORANGE, linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the updated prediction is the input of the next stage", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**One gradient boosting stage takes the prediction after the earlier trees in and passes a shrunk update on.** The pseudo-residuals are computed from that prediction, a tree is fitted to them, and only the learning rate times its answer is added.

__Step 24:__ Compute residuals for six toy responses, fit the first correction tree, and add half its answer.

In [ ]:
toy_response = np.array([2., 3., 2., 8., 9., 8.])
toy_position = np.arange(6).reshape(-1, 1)
toy_initial = np.full(6, toy_response.mean())
toy_residual = toy_response - toy_initial
toy_correction = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy_position, toy_residual)
toy_updated = toy_initial + 0.5 * toy_correction.predict(toy_position)
print(pd.DataFrame({"response": toy_response, "initial": toy_initial, "residual": toy_residual, "correction": toy_correction.predict(toy_position), "updated": toy_updated}).round(3).to_string(index=False))

For squared error, start at the response mean $F_0$ and compute residual $r_i=y_i-F_0$. Fit correction tree $h$ to those residuals, then set $F_1(x)=F_0+\eta h(x)$. Here $i$ indexes rows, $y_i$ is an observed response, $x$ is an input and $\eta$ is the learning rate. For a general loss $L$, the residual is replaced by the pseudo-residual $r_i=-\partial L(y_i,F(x_i))/\partial F(x_i)$, the negative slope of the loss with respect to the prediction $F(x_i)$ of the earlier trees; the half squared error $L=\tfrac{1}{2}(y-F)^2$ gives back $y_i-F(x_i)$.

__Step 25:__ Draw the fitted toy residual tree.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plot_tree(toy_correction, feature_names=["toy position"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("Gradient boosting: first correction")
plt.tight_layout()
plt.show()

**The correction tree splits the six rows at position 2.5 and answers -3.0 on the left and 3.0 on the right.** Adding half of each answer to the starting mean 5.333 gives updated values of 3.833 and 6.833, half of the way towards the left and right response means.

Here stage 0 starts from an ordinary shallow classification tree. Its leaf's share of label 1 gives the starting probability; the log-odds gives the starting raw score. Each later stage fits a regression tree to the remaining probability error and adds its shrunk answer to the raw score. The tables show every row's numbers, which scikit-learn reproduces.

__Step 26:__ Fit a regular classification tree to ten toy rows and draw it: its leaf probabilities are the starting prediction.

In [ ]:
# Ten toy rows with one input, the position, and a binary label.
walk_x = np.arange(10).reshape(-1, 1)
walk_label = np.array([0, 1, 0, 0, 0, 1, 1, 0, 1, 1])
# Stage 0 is a regular shallow classification tree fitted to the labels.
walk_start = DecisionTreeClassifier(max_depth=1, random_state=random_state).fit(walk_x, walk_label)
# Its leaf probabilities, the share of label 1 in the leaf of each row, are the starting prediction.
walk_start_probability = walk_start.predict_proba(walk_x)[:, 1]
# Draw the tree: each leaf shows its class counts and the class it predicts.
fig, ax = plt.subplots(figsize=(7, 3))
plot_tree(walk_start, feature_names=["toy position"], class_names=["0", "1"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("stage 0: a regular classification tree")
plt.tight_layout()
plt.show()
print("starting probability of each row:", walk_start_probability.round(3))

**The stage 0 tree gives every row of its left leaf the probability 0.2 and every row of its right leaf 0.8.** These shares of label 1 are the starting prediction, and their log-odds, -1.386 and 1.386, are the raw scores that stage 1 starts from.

__Step 27:__ Work out the first gradient boosting stage by hand for every row: raw score, probability, pseudo-residual, leaf value and update.

In [ ]:
# The learning rate shrinks every tree's answer before it is added.
walk_rate = 0.5
# Raw score = log-odds of the probability; stage 1 starts from the log-odds of the stage 0 leaf probabilities.
walk_raw = np.log(walk_start_probability / (1 - walk_start_probability))
walk_probability = walk_start_probability
# Pseudo-residual: the label minus the probability, which is minus the slope of the log loss at the raw score.
walk_residual = walk_label - walk_probability
# A regression tree is fitted to the pseudo-residuals, not to the labels.
walk_tree = DecisionTreeRegressor(max_depth=3, random_state=random_state).fit(walk_x, walk_residual)
# scikit-learn replaces each leaf value by one Newton step: the sum of the pseudo-residuals in the leaf over the sum of p(1-p) there.
walk_leaf = walk_tree.apply(walk_x)
walk_value = np.array([walk_residual[walk_leaf == leaf].sum() / (walk_probability * (1 - walk_probability))[walk_leaf == leaf].sum() for leaf in walk_leaf])
# The new raw score adds the learning rate times the leaf value; the probability follows from it.
walk_new_raw = walk_raw + walk_rate * walk_value
walk_new_probability = 1 / (1 + np.exp(-walk_new_raw))
walk_trees = [walk_tree]
walk_tables = [pd.DataFrame({"label": walk_label, "raw score before": walk_raw, "probability before": walk_probability,
    "pseudo-residual": walk_residual, "leaf value": walk_value, "learning rate x leaf value": walk_rate * walk_value,
    "raw score after": walk_new_raw, "probability after": walk_new_probability})]
display(walk_tables[0].round(3))

**Row 1 has label 1 and starting probability 0.2, so its pseudo-residual is 0.8 and its leaf value of 5.0 lifts its raw score from -1.386 to 1.114 after the learning rate of 0.5.** Row 7 has label 0 and probability 0.8, so its pseudo-residual is -0.8 and its raw score falls to -1.114. Rows 2 to 6 share one leaf whose value is -0.25, which is not the plain mean of their pseudo-residuals, -0.2 three times and 0.2 twice, because the sum is divided by the sum of $p(1-p)$.

One row's loss is $L=-yF+\log(1+e^{F})$, where $y$ is its label, coded 0 or 1, and $F$ its raw score. With $p=1/(1+e^{-F})$, the slope is $p-y$, so pseudo-residual $y-p$ is its negative. A row labelled 1 with low probability has a large positive pseudo-residual: raising $F$ lowers its loss quickly. Each stage adds the learning rate times the leaf value. This value is a Newton step, not the pseudo-residual mean: scikit-learn divides their sum within the leaf by the sum of $p(1-p)$, the loss curvature.

__Step 28:__ Draw the loss of each row against its raw score, the tangent at the raw score before stage 1 and the move made by the stage.

In [ ]:
# Log loss of one row against its raw score F: L = -y F + log(1 + exp(F)), one curve for each label.
walk_scores = np.linspace(-4, 4, 200)
walk_curves = {1: -walk_scores + np.log1p(np.exp(walk_scores)), 0: np.log1p(np.exp(walk_scores))}
# The loss of every row before and after the stage, from the raw scores of the table above.
walk_loss_before = -walk_label * walk_raw + np.log1p(np.exp(walk_raw))
walk_loss_after = -walk_label * walk_new_raw + np.log1p(np.exp(walk_new_raw))
fig, ax = plt.subplots(figsize=(8, 5))
for label, colour in ((1, PLOT_ORANGE), (0, PLOT_BLUE)):
    ax.plot(walk_scores, walk_curves[label], color=colour, linewidth=2, label=f"loss of a row with label {label}")
# For each row: a dot at its raw score before the stage, the tangent there, and an arrow to its raw score after the stage.
for row in range(len(walk_label)):
    colour = PLOT_ORANGE if walk_label[row] == 1 else PLOT_BLUE
    ax.scatter(walk_raw[row], walk_loss_before[row], color=colour, zorder=4)
    ax.plot([walk_raw[row] - 0.6, walk_raw[row] + 0.6], [walk_loss_before[row] + 0.6 * walk_residual[row], walk_loss_before[row] - 0.6 * walk_residual[row]], color="gray", linewidth=1)
    ax.annotate("", xy=(walk_new_raw[row], walk_loss_after[row]), xytext=(walk_raw[row], walk_loss_before[row]), arrowprops=dict(arrowstyle="->", color=colour))
ax.set(xlabel="raw score F (log-odds)", ylabel="loss of the row", title="stage 1: the tangent slope is minus the pseudo-residual, the arrow is the update")
ax.legend()
plt.tight_layout()
plt.show()
print(f"total loss of the ten rows before stage 1 {walk_loss_before.sum():.4f}, after stage 1 {walk_loss_after.sum():.4f}")

**The total loss of the ten rows falls from 5.0040 to 2.0416 in the first stage.** Rows 5 and 6, labelled 1, show that a leaf moves its rows together: their probability goes from 0.8 to 0.779, a slightly higher loss for them, while rows with a large pseudo-residual move furthest down their curves.

__Step 29:__ Work out the second and third stages the same way.

In [ ]:
# Stages 2 and 3 repeat stage 1; each starts from the raw score the previous stage ended with.
for stage in tqdm((2, 3), desc='stage', leave=True, position=0):
    walk_raw, walk_probability = walk_new_raw, walk_new_probability
    walk_residual = walk_label - walk_probability
    walk_tree = DecisionTreeRegressor(max_depth=3, random_state=random_state).fit(walk_x, walk_residual)
    walk_leaf = walk_tree.apply(walk_x)
    walk_value = np.array([walk_residual[walk_leaf == leaf].sum() / (walk_probability * (1 - walk_probability))[walk_leaf == leaf].sum() for leaf in walk_leaf])
    walk_new_raw = walk_raw + walk_rate * walk_value
    walk_new_probability = 1 / (1 + np.exp(-walk_new_raw))
    walk_trees.append(walk_tree)
    walk_tables.append(pd.DataFrame({"label": walk_label, "raw score before": walk_raw, "probability before": walk_probability,
        "pseudo-residual": walk_residual, "leaf value": walk_value, "learning rate x leaf value": walk_rate * walk_value,
        "raw score after": walk_new_raw, "probability after": walk_new_probability}))
    print(f"stage {stage}")
    display(walk_tables[-1].round(3))

**Row 7, labelled 0, keeps falling: its probability is 0.8 before stage 1 and then 0.247, 0.145 and 0.086 after stages 1, 2 and 3.** Row 1, labelled 1, rises through 0.753, 0.855 and 0.914 from a start of 0.2.

__Step 30:__ Draw the regression tree fitted at each stage.

In [ ]:
# One full-width figure per stage: the regression tree fitted to that stage's pseudo-residuals. Its boxes show the mean pseudo-residual of each leaf.
for stage, tree in enumerate(walk_trees, start=1):
    fig, axis = plt.subplots(figsize=(12, 5))
    plot_tree(tree, feature_names=["toy position"], filled=True, rounded=True, precision=2, fontsize=9, ax=axis)
    axis.set_title(f"stage {stage}: regression tree on the pseudo-residuals")
    plt.tight_layout()
    plt.show()

**Each figure shows the regression tree of one stage, and its leaf boxes show the mean pseudo-residual of the rows in the leaf.** The update in the tables uses the rescaled leaf value, not this mean.

__Step 31:__ Check each hand stage against scikit-learn's `GradientBoostingClassifier`, started from the same stage 0 tree.

In [ ]:
# scikit-learn started from the same stage 0 tree, with the same learning rate and tree depth, must reach the same probabilities.
walk_library = GradientBoostingClassifier(init=walk_start, n_estimators=3, learning_rate=walk_rate, max_depth=3, random_state=random_state).fit(walk_x, walk_label)
walk_staged = [probability[:, 1] for probability in walk_library.staged_predict_proba(walk_x)]
# Compare the probability after each stage with the hand table, within a stated tolerance of 1e-8.
for stage, (table, library) in enumerate(zip(walk_tables, walk_staged), start=1):
    assert np.allclose(table["probability after"], library, rtol=0, atol=1e-8), stage
print("the three hand stages equal staged_predict_proba within 1e-8; largest difference",
      max(np.abs(table["probability after"] - library).max() for table, library in zip(walk_tables, walk_staged)))

**The three hand stages match the staged probabilities of scikit-learn's `GradientBoostingClassifier` to within 1e-8, with a largest difference of 2.2e-16.** Starting the library model from the same stage 0 tree is what makes the two start from the same probabilities.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html'>sklearn.ensemble.GradientBoostingClassifier(loss='log_loss', learning_rate=0.1, n_estimators=100, subsample=1.0, min_samples_split=2, min_samples_leaf=1, max_depth=3, max_features=None, ccp_alpha=0.0, random_state=None)</a>

**Definition:**
Adds one regression tree per stage, fitted to the negative gradient of the loss at the current raw scores, and sums the trees into a raw score that a link converts to a probability.

**Main Parameters:**
- `loss`: loss whose gradient each tree fits.
- `learning_rate`, `n_estimators`: tree multiplier and number of trees.
- `subsample`: fraction of rows drawn for each tree.
- `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `ccp_alpha`: growth controls of each tree.
- `random_state`: seed of row and column draws.

**Main Attributes (after fitting):**
- `estimators_`: the fitted regression trees.
- `init_`: the model that gives the starting raw score.
- `train_score_`: the training loss after each stage.

**Main Methods:**
- `fit(X, y)`: add the trees in sequence.
- `predict_proba(X)`: class probabilities.
- `decision_function(X)`: the raw score.
- `staged_predict(X)`: the prediction after each stage.
</div>

__Step 32:__ Score the GradientBoostingClassifier reference on the ten prepared training folds.

In [ ]:
gb_reference = GradientBoostingClassifier(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
gb_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
gb_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 33:__ Fit the GradientBoostingClassifier reference on all the training rows.

In [ ]:
gb_default = clone(gb_reference).fit(Z_train, y_train)
gb_default_configuration = " ".join(str(gb_reference).split())
gb_complete_rows = []

__Step 34:__ Log the GradientBoostingClassifier reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_default_configuration, "train": gb_default_train, "validation": gb_default_validation,
    "model": gb_default},
    Z_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Gradient boosting settings</font> <a class="anchor" id="gb_settings"></a>
[Back to TOC](#toc)

Each sweep changes one gradient boosting setting from the default model and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; the rate and stage-count interaction is measured separately. The other tree settings, `min_samples_leaf`, `min_samples_split`, `max_features` and `ccp_alpha`, were taught with decision trees and stay at their reference values here.

| Parameter | Meaning |
| --- | --- |
| `n_estimators` | Number of regression trees. |
| `learning_rate` | Multiplier on each tree's answer. |
| `max_depth` | Splits allowed on a path of a stage tree. |
| `subsample` | Fraction of rows drawn for each tree. |
| `loss` | Loss whose negative gradient each tree fits. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)

__Step 35:__ Change only gradient boosting `n_estimators` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `gradient boosting: n_estimators`

Each stage adds a regression tree to the current prediction; the default is `100`. More stages are expected to reduce training loss and increase runtime, but validation may stop improving. The learning rate controls the size of each correction. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
gb_n_estimators_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
gb_n_estimators_rows = []
gb_stage_count = max(gb_n_estimators_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
gb_train_by_fold = np.zeros((len(prepared), gb_stage_count))
gb_validation_by_fold = np.zeros((len(prepared), gb_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(n_estimators=gb_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="GradientBoostingClassifier staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(gb_stage_count):
        used = min(stage, len(on_fit) - 1)
        gb_train_by_fold[position, stage] = f1_score(fold["y_fit"], on_fit[used])
        gb_validation_by_fold[position, stage] = f1_score(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(gb_n_estimators_values[:-1], desc='gb n estimators values[:-1]', leave=True, position=0):
    fresh = clone(gb_reference).set_params(n_estimators=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.array_equal(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", gb_n_estimators_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(gb_n_estimators_values, desc='GradientBoostingClassifier.n_estimators', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(n_estimators=value)
    train_scores = list(gb_train_by_fold[:, value - 1])
    validation_scores = list(gb_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    gb_n_estimators_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
gb_n_estimators_sweep = pd.DataFrame(gb_n_estimators_rows)
gb_complete_rows.extend(gb_n_estimators_rows)
gb_n_estimators_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_n_estimators_winner = gb_n_estimators_sweep.loc[gb_n_estimators_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_n_estimators_winner["configuration"], "train": gb_n_estimators_winner["train"],
    "validation": gb_n_estimators_winner["validation"], "model": gb_n_estimators_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 rises from 0.7532 with one stage to 0.8584 at 30 stages and peaks at 0.8636 at 110 stages, against 0.8631 at 150, while training F1 rises from 0.7532 to 0.9069 and stays above validation F1 from 5 stages on.**

__Step 36:__ Plot training and validation F1 against the number of gradient boosting stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
gb_stages = np.arange(1, gb_stage_count + 1)
ax.plot(gb_stages, gb_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(gb_stages, gb_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(gb_n_estimators_sweep["setting"], gb_n_estimators_sweep["train_F1"], color=PLOT_ORANGE, zorder=3)
ax.scatter(gb_n_estimators_sweep["setting"], gb_n_estimators_sweep["cv_F1"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="n_estimators", ylabel="F1", title="GradientBoostingClassifier: train against validation", ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.show()

__Step 37:__ Change only gradient boosting `learning_rate` on the fixed training folds, with the sweep list `[0.03, 0.1, 0.3]`.

### `gradient boosting: learning_rate`

The learning rate multiplies each new tree's correction; the default is `0.1`. A smaller rate is expected to need more stages, while a larger rate can reach a useful fit sooner and then overfit. Validation, rather than an equal rate-times-stage product, decides the useful combination. The candidate list is `[0.03, 0.1, 0.3]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_learning_rate_values = [0.03, 0.1, 0.3]
# One row per value will hold its scores, its seconds and its fitted model.
gb_learning_rate_rows = []
for value in tqdm(gb_learning_rate_values, desc="GradientBoostingClassifier.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_learning_rate_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_learning_rate_sweep = pd.DataFrame(gb_learning_rate_rows)
gb_complete_rows.extend(gb_learning_rate_rows)
gb_learning_rate_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_learning_rate_winner = gb_learning_rate_sweep.loc[gb_learning_rate_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_learning_rate_winner["configuration"], "train": gb_learning_rate_winner["train"],
    "validation": gb_learning_rate_winner["validation"], "model": gb_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**A rate of 0.1 gives the best validation F1, 0.8628, between 0.8591 at 0.03 and 0.8564 at 0.3.** At 0.3 training F1 reaches 0.9412 while validation falls.

__Step 38:__ Change only gradient boosting `max_depth` on the fixed training folds, with the sweep list `[1, 3, 5]`.

### `gradient boosting: max_depth`

Depth limits the number of splits along each tree path and therefore which feature interactions one stage can model. The default is `3`. Deeper trees are expected to fit residual structure more closely, increasing both runtime and the risk of fitting noise. The candidate list is `[1, 3, 5]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_max_depth_values = [1, 3, 5]
# One row per value will hold its scores, its seconds and its fitted model.
gb_max_depth_rows = []
for value in tqdm(gb_max_depth_values, desc="GradientBoostingClassifier.max_depth", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(max_depth=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_max_depth_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_max_depth_sweep = pd.DataFrame(gb_max_depth_rows)
gb_complete_rows.extend(gb_max_depth_rows)
gb_max_depth_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_max_depth_winner = gb_max_depth_sweep.loc[gb_max_depth_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_max_depth_winner["configuration"], "train": gb_max_depth_winner["train"],
    "validation": gb_max_depth_winner["validation"], "model": gb_max_depth_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8419 at depth 1, 0.8628 at depth 3 and 0.8616 at depth 5.** Training F1 climbs from 0.8497 to 0.8938 and 0.9563 across the three depths.

__Step 39:__ Change only gradient boosting `subsample` on the fixed training folds, with the sweep list `[0.6, 0.8, 1.0]`.

### `gradient boosting: subsample`

Each stage samples this fraction of training rows without replacement; the default `1.0` uses every row. A smaller fraction is expected to lower runtime and make the stage trees differ. It can reduce variance while increasing bias. The candidate list is `[0.6, 0.8, 1.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_subsample_values = [0.6, 0.8, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
gb_subsample_rows = []
for value in tqdm(gb_subsample_values, desc="GradientBoostingClassifier.subsample", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(subsample=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_subsample_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_subsample_sweep = pd.DataFrame(gb_subsample_rows)
gb_complete_rows.extend(gb_subsample_rows)
gb_subsample_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_subsample_winner = gb_subsample_sweep.loc[gb_subsample_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_subsample_winner["configuration"], "train": gb_subsample_winner["train"],
    "validation": gb_subsample_winner["validation"], "model": gb_subsample_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Drawing 60 percent or 80 percent of the rows per stage leaves validation F1 at 0.8625 and 0.8636, against 0.8628 with every row.**

__Step 40:__ Change only gradient boosting `loss` on the fixed training folds, with the sweep list `['log_loss', 'exponential']`.

### `gradient boosting: loss`

Loss defines the negative gradient fitted by each stage. The default `log_loss` gives the pseudo-residual y minus the current positive-class probability. Exponential loss gives confidently wrong predictions greater influence, so difficult or noisy rows are expected to steer subsequent trees more strongly. The candidate list is `['log_loss', 'exponential']`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_loss_values = ['log_loss', 'exponential']
# One row per value will hold its scores, its seconds and its fitted model.
gb_loss_rows = []
for value in tqdm(gb_loss_values, desc="GradientBoostingClassifier.loss", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(loss=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_loss_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_loss_sweep = pd.DataFrame(gb_loss_rows)
gb_complete_rows.extend(gb_loss_rows)
gb_loss_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_loss_winner = gb_loss_sweep.loc[gb_loss_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_loss_winner["configuration"], "train": gb_loss_winner["train"],
    "validation": gb_loss_winner["validation"], "model": gb_loss_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The exponential loss scores 0.8637 against 0.8628 for the log loss.**

__Step 41:__ Plot training and validation F1 for every other swept gradient boosting setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
gb_settings_tables = {"learning_rate": (gb_learning_rate_sweep, gb_learning_rate_values), "max_depth": (gb_max_depth_sweep, gb_max_depth_values), "subsample": (gb_subsample_sweep, gb_subsample_values), "loss": (gb_loss_sweep, gb_loss_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), gb_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="F1", ylim=(0, 1.05))
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(gb_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**Across the four panels validation F1 stays between 0.8419 and 0.8637, while training F1 reaches 0.9412 at a learning rate of 0.3 and 0.9563 at depth 5.** In the other two panels training F1 stays between 0.8878 and 0.8959, so only the learning rate and the depth open a visible gap between the two lines.

__Step 42:__ Draw the validation F1 of gradient boosting at every stage for four learning rates.

### `gradient boosting: learning rate and stage count`

Smaller corrections often need more stages, but noise, overfitting and early termination can break that pattern. The first study plots actual fitted stages against validation score at each learning rate, with an explicit larger stage budget. The second enables a stopping criterion; validation determines whether its chosen count helps.

In [ ]:
# Four learning rates, each with its own staged validation curve (mean over the ten folds).
gb_curve_rates = [0.03, 0.1, 0.3, 1.0]
gb_rate_curves = {}
# Fit every fold once per rate at the largest stage count, then score the prediction after each stage: one fit gives the whole curve.
for rate in tqdm(gb_curve_rates, desc="GradientBoostingClassifier learning rates", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(learning_rate=rate, n_estimators=300).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    gb_rate_curves[rate] = np.mean([[f1_score(fold["y_score"], on_score) for on_score in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)], axis=0)
# Draw the curves; the dot marks the stage with the best validation score of each rate.
fig, ax = plt.subplots(figsize=(7, 5))
for rate, curve in gb_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter(np.argmax(curve) + 1, curve.max(), zorder=3)
ax.set(xlabel="n_estimators", ylabel="validation F1", title="GradientBoostingClassifier: validation by stage at four learning rates")
ax.legend()
plt.tight_layout()
plt.show()
# Table of the same curves: the best stage, the stage where each curve first comes close to its best, and that best score.
pd.DataFrame({"best stage": [int(np.argmax(curve)) + 1 for curve in gb_rate_curves.values()],
    "stages to come within 1 percent of the best": [int(np.argmax(curve >= 0.99 * curve.max())) + 1 for curve in gb_rate_curves.values()],
    "best validation F1": [curve.max() for curve in gb_rate_curves.values()]}, index=pd.Index(gb_curve_rates, name="learning_rate")).round(4)

**A rate of 0.03 reaches validation F1 0.8641 at stage 291, a rate of 0.1 reaches 0.8643 at stage 121, a rate of 0.3 reaches 0.8622 at stage 58 and a rate of 1.0 reaches 0.8532 at stage 7.** Within 1 percent of its best, the rate 0.03 needs 77 stages, the rate 0.1 needs 24, the rate 0.3 needs 9 and the rate 1.0 needs 3, so shrinkage buys more stages for the same fit.

__Step 43:__ Stop each gradient boosting fit with its convergence criterion and count the stages it used.

In [ ]:
# With a convergence criterion each fit stops by itself, so the learning rate decides how many stages are fitted.
gb_stop_rows = []
for rate in tqdm(gb_curve_rates, desc="GradientBoostingClassifier early stopping", unit="rate"):
    started = time.perf_counter()
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(learning_rate=rate, n_estimators=1000, n_iter_no_change=10, validation_fraction=0.1).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    gb_stop_rows.append({"learning_rate": rate, "stages fitted": np.mean([fitted.n_estimators_ for fitted in fits]),
        "validation F1": np.mean([f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]),
        "seconds": time.perf_counter() - started})
pd.DataFrame(gb_stop_rows).set_index("learning_rate").round(4)

**With the criterion on, the fit stops after 287.9 stages on average over the folds at a rate of 0.03, 105.6 at 0.1, 44.0 at 0.3 and 13.8 at 1.0.** The validation F1 is 0.8633, 0.8627, 0.8607 and 0.8494, so the learning rate decides when training stops and how far it gets.

__Step 44:__ Select the best tested gradient boosting configuration from the existing sweep rows by validation score.

In [ ]:
gb_complete = pd.DataFrame(gb_complete_rows)
gb_best = gb_complete.loc[gb_complete["cv_F1"].idxmax()]
gb_best_parameters = gb_best["parameters"]
gb_best_configuration = clone(gb_reference).set_params(**gb_best_parameters)
print("best tested gradient boosting configuration:", gb_best["configuration"])

**The best complete gradient boosting configuration measured uses the exponential loss, with validation F1 0.8637.**

# <font color='#E8800A'>Histogram gradient boosting</font> <a class="anchor" id="hist"></a>
[Back to TOC](#toc)

Histogram boosting groups numeric values into bins before considering splits. It retains sequential loss corrections while reducing the number of candidate thresholds. [Histogram boosting guide](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)

__Step 45:__ Group six toy values into three bins and count the rows offered to each bin.

In [ ]:
toy_values = np.array([1., 2., 3., 7., 8., 9.])
toy_edges = np.array([3.5, 7.5])
toy_bins = np.searchsorted(toy_edges, toy_values)
print(pd.DataFrame({"value": toy_values, "bin": toy_bins}).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(np.arange(3), np.bincount(toy_bins, minlength=3), color=PLOT_BLUE)
ax.set(xlabel="Bin", ylabel="Training rows", title="Three toy histogram bins")
plt.tight_layout()
plt.show()

**The values 1, 2 and 3 fall in bin 0, the value 7 in bin 1 and the values 8 and 9 in bin 2, so the three bins hold 3, 1 and 2 rows.**

__Step 46:__ Draw the toy values on a line with the candidate cuts of an exact tree and of histogram boosting.

In [ ]:
# Top: an exact tree may cut halfway between any two neighbouring values. Bottom: histogram boosting may cut only at the bin edges.
bin_midpoints = (toy_values[:-1] + toy_values[1:]) / 2
bin_colours = [PLOT_BLUE, PLOT_ORANGE, "#4C4C4C"]
fig, axes = plt.subplots(2, 1, figsize=(9, 3.8), sharex=True)
for axis, thresholds, title in zip(axes, [bin_midpoints, toy_edges], ["exact tree", "histogram boosting"]):
    axis.scatter(toy_values, np.zeros(len(toy_values)), c=[bin_colours[bin_index] for bin_index in toy_bins], s=140, zorder=3)
    for threshold in thresholds:
        axis.axvline(threshold, color="gray", linestyle="--")
    axis.set(title=f"{title}: {len(thresholds)} candidate cuts", yticks=[])
    print(f"{title}: {len(thresholds)} candidate cuts at {np.round(thresholds, 1).tolist()}")
# Shade the bins on the bottom panel: every value inside one shade is treated alike when a cut is chosen.
for (left, right), shade in zip(zip(np.r_[toy_values.min() - 0.5, toy_edges], np.r_[toy_edges, toy_values.max() + 0.5]), bin_colours):
    axes[1].axvspan(left, right, color=shade, alpha=0.15)
axes[1].set_xlabel("raw value")
plt.tight_layout()
plt.show()

**Binning cuts the exact tree's candidate thresholds down to the bin edges.** Every value inside one bin is treated alike when a cut is chosen, which makes the search for a split cheaper; the stages then proceed as in gradient boosting.

__Step 47:__ Draw one histogram boosting stage as a flow: binned input, then a gradient boosting stage.

In [ ]:
# Each box is a patch on a blank axis: the binned input of histogram boosting and one stage that then proceeds as in gradient boosting. Arrows show the order inside one stage.
stage_boxes = ['columns cut into bins\n(once, before stage 1)', 'gradient and hessian\nof the loss per row', 'tree grown leaf by leaf\non the bin sums', 'F + learning rate x h:\na shrunk update']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor="#4C4C4C", linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the updated prediction is the input of the next stage; the bins are not recomputed", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**Histogram boosting bins the columns once, and every stage after that is a gradient boosting stage.** The tree is grown leaf by leaf on sums of the gradient and hessian within the bins, and its shrunk answer updates the prediction.

__Step 48:__ Draw the three loops side by side: what goes in, what is fitted and what is passed on.

In [ ]:
# Rows: the three algorithms. Columns: what goes into a stage, what is fitted, and what is passed on.
loops = [("AdaBoost", PLOT_BLUE, ["row weights", "a learner fitted\nto the weighted rows", "new row weights and\na stage weight"]),
         ("gradient boosting", PLOT_ORANGE, ["prediction after the\nearlier trees", "a tree fitted to the\npseudo-residuals", "prediction plus a\nshrunk tree"]),
         ("histogram boosting", "#4C4C4C", ["binned columns, then the prediction\nafter the earlier trees", "a tree grown leaf by leaf\non bin sums", "prediction plus a\nshrunk tree"])]
fig, ax = plt.subplots(figsize=(12, 4.6))
for column, header in enumerate(["goes in", "is fitted", "is passed on"]):
    ax.text(column * 4.4 + 1.7, 4.95, header, ha="center", fontsize=11, fontweight="bold")
for row, (name, colour, labels) in enumerate(loops):
    height = (2 - row) * 1.65
    ax.text(-0.3, height + 0.6, name, ha="right", va="center", fontsize=11, fontweight="bold")
    for column, label in enumerate(labels):
        ax.add_patch(FancyBboxPatch((column * 4.4, height), 3.4, 1.2, boxstyle="round,pad=0.06", facecolor="white", edgecolor=colour, linewidth=2))
        ax.text(column * 4.4 + 1.7, height + 0.6, label, ha="center", va="center", fontsize=10)
        if column:
            ax.annotate("", xy=(column * 4.4, height + 0.6), xytext=(column * 4.4 - 1.0, height + 0.6), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.set(xlim=(-3.4, 12.8), ylim=(-0.2, 5.3))
ax.axis("off")
plt.tight_layout()
plt.show()

**Read the rows from left to right and compare what each loop passes on.** AdaBoost passes row weights, gradient boosting passes a prediction whose residuals the next tree fits, and histogram boosting passes the same prediction while working on binned columns.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html'>sklearn.ensemble.HistGradientBoostingClassifier(loss='log_loss', learning_rate=0.1, max_iter=100, max_leaf_nodes=31, max_depth=None, min_samples_leaf=20, l2_regularization=0.0, max_features=1.0, max_bins=255, early_stopping='auto', random_state=None, class_weight=None)</a>

**Definition:**
Bins every numeric column into at most 255 integer bins, then adds one leaf-wise grown tree per iteration, fitted to the gradient and hessian of the loss.

**Main Parameters:**
- `max_iter`, `learning_rate`: number of iterations and leaf-value multiplier.
- `max_leaf_nodes`, `max_depth`, `min_samples_leaf`: tree size controls.
- `l2_regularization`: penalty that shrinks leaf values.
- `max_bins`, `max_features`: threshold resolution and columns per split.
- `early_stopping`: stops adding iterations when a validation score stalls.
- `class_weight`: per-class weight on the loss.

**Main Attributes (after fitting):**
- `n_iter_`: iterations actually fitted.
- `train_score_`: the training score after each iteration when scoring is on.
- `n_trees_per_iteration_`: trees added per iteration.

**Main Methods:**
- `fit(X, y)`: bin the columns and add the trees.
- `predict_proba(X)`: class probabilities.
- `staged_predict(X)`: the prediction after each iteration.
</div>

__Step 49:__ Score the HistGradientBoostingClassifier reference on the ten prepared training folds.

In [ ]:
hist_reference = HistGradientBoostingClassifier(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
hist_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
hist_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 50:__ Fit the HistGradientBoostingClassifier reference on all the training rows.

In [ ]:
hist_default = clone(hist_reference).fit(Z_train, y_train)
hist_default_configuration = " ".join(str(hist_reference).split())
hist_complete_rows = []

__Step 51:__ Log the HistGradientBoostingClassifier reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_default_configuration, "train": hist_default_train, "validation": hist_default_validation,
    "model": hist_default},
    Z_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Histogram boosting settings</font> <a class="anchor" id="hist_settings"></a>
[Back to TOC](#toc)

Each sweep changes one histogram boosting setting from the reference, whose early stopping is off, and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; interactions need the later joint search. The other tree settings, `max_depth`, `min_samples_leaf`, `max_features` and `class_weight`, were taught with decision trees and stay at their reference values here.

| Parameter | Meaning |
| --- | --- |
| `max_iter` | Number of boosting stages. |
| `learning_rate` | Multiplier on each tree's leaf values. |
| `max_leaf_nodes` | Leaves allowed in a tree. |
| `l2_regularization` | Penalty that shrinks leaf values. |
| `max_bins` | Bins per numeric column. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html)

__Step 52:__ Change only histogram boosting `max_iter` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `histogram boosting: max_iter`

One iteration adds a tree; the default is `100`. The default `early_stopping="auto"` activates only above 10000 training rows, so it stays off for these smaller folds. More iterations are expected to fit training rows more closely, with validation deciding whether the extra cost helps. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
hist_max_iter_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
hist_max_iter_rows = []
hist_stage_count = max(hist_max_iter_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
hist_train_by_fold = np.zeros((len(prepared), hist_stage_count))
hist_validation_by_fold = np.zeros((len(prepared), hist_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).set_params(max_iter=hist_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="HistGradientBoostingClassifier staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(hist_stage_count):
        used = min(stage, len(on_fit) - 1)
        hist_train_by_fold[position, stage] = f1_score(fold["y_fit"], on_fit[used])
        hist_validation_by_fold[position, stage] = f1_score(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(hist_max_iter_values[:-1], desc='hist max iter values[:-1]', leave=True, position=0):
    fresh = clone(hist_reference).set_params(max_iter=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.array_equal(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", hist_max_iter_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(hist_max_iter_values, desc='HistGradientBoostingClassifier.max_iter', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_iter=value)
    train_scores = list(hist_train_by_fold[:, value - 1])
    validation_scores = list(hist_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    hist_max_iter_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
hist_max_iter_sweep = pd.DataFrame(hist_max_iter_rows)
hist_complete_rows.extend(hist_max_iter_rows)
hist_max_iter_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_iter_winner = hist_max_iter_sweep.loc[hist_max_iter_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_iter_winner["configuration"], "train": hist_max_iter_winner["train"],
    "validation": hist_max_iter_winner["validation"], "model": hist_max_iter_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.7532 after one stage, peaks at 0.8665 at 30 stages and then falls to 0.8596 at 150 stages, while training F1 climbs from 0.7532 to 0.9948.** The two lines separate as stages are added.

__Step 53:__ Plot training and validation F1 against the number of histogram boosting stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
hist_stages = np.arange(1, hist_stage_count + 1)
ax.plot(hist_stages, hist_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(hist_stages, hist_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(hist_max_iter_sweep["setting"], hist_max_iter_sweep["train_F1"], color=PLOT_ORANGE, zorder=3)
ax.scatter(hist_max_iter_sweep["setting"], hist_max_iter_sweep["cv_F1"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="max_iter", ylabel="F1", title="HistGradientBoostingClassifier: train against validation", ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.show()

__Step 54:__ Change only histogram boosting `learning_rate` on the fixed training folds, with the sweep list `[0.03, 0.1, 0.3]`.

### `histogram boosting: learning_rate`

The learning rate scales each new tree's leaf values; the default is `0.1`. A smaller rate is expected to need more iterations, while a larger rate reaches a close training fit sooner. Neither guarantees better validation error. The candidate list is `[0.03, 0.1, 0.3]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_learning_rate_values = [0.03, 0.1, 0.3]
# One row per value will hold its scores, its seconds and its fitted model.
hist_learning_rate_rows = []
for value in tqdm(hist_learning_rate_values, desc="HistGradientBoostingClassifier.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_learning_rate_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_learning_rate_sweep = pd.DataFrame(hist_learning_rate_rows)
hist_complete_rows.extend(hist_learning_rate_rows)
hist_learning_rate_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_learning_rate_winner = hist_learning_rate_sweep.loc[hist_learning_rate_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_learning_rate_winner["configuration"], "train": hist_learning_rate_winner["train"],
    "validation": hist_learning_rate_winner["validation"], "model": hist_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The smallest rate scores best: validation F1 is 0.8668 at 0.03, 0.8626 at 0.1 and 0.8554 at 0.3, with training F1 reaching 1.0000 at 0.3.**

__Step 55:__ Change only histogram boosting `max_leaf_nodes` on the fixed training folds, with the sweep list `[7, 15, 31]`.

### `histogram boosting: max_leaf_nodes`

The tree repeatedly splits the leaf giving the largest loss reduction until reaching this cap. The default is `31`. More leaves are expected to capture finer structure at greater cost, with a greater chance of fitting noise. The candidate list is `[7, 15, 31]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_max_leaf_nodes_values = [7, 15, 31]
# One row per value will hold its scores, its seconds and its fitted model.
hist_max_leaf_nodes_rows = []
for value in tqdm(hist_max_leaf_nodes_values, desc="HistGradientBoostingClassifier.max_leaf_nodes", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_leaf_nodes=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_max_leaf_nodes_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_max_leaf_nodes_sweep = pd.DataFrame(hist_max_leaf_nodes_rows)
hist_complete_rows.extend(hist_max_leaf_nodes_rows)
hist_max_leaf_nodes_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_leaf_nodes_winner = hist_max_leaf_nodes_sweep.loc[hist_max_leaf_nodes_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_leaf_nodes_winner["configuration"], "train": hist_max_leaf_nodes_winner["train"],
    "validation": hist_max_leaf_nodes_winner["validation"], "model": hist_max_leaf_nodes_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Fifteen leaves per tree give the best validation F1, 0.8665, against 0.8649 at 7 and 0.8626 at 31.**

__Step 56:__ Change only histogram boosting `l2_regularization` on the fixed training folds, with the sweep list `[0.0, 1.0, 10.0]`.

### `histogram boosting: l2_regularization`

This penalty enlarges the denominator used to calculate a leaf value, shrinking its correction. The default is `0.0`, with no penalty. A larger value is expected to weaken corrections most in leaves with few rows or low loss curvature. The candidate list is `[0.0, 1.0, 10.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_l2_regularization_values = [0.0, 1.0, 10.0]
# One row per value will hold its scores, its seconds and its fitted model.
hist_l2_regularization_rows = []
for value in tqdm(hist_l2_regularization_values, desc="HistGradientBoostingClassifier.l2_regularization", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(l2_regularization=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_l2_regularization_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_l2_regularization_sweep = pd.DataFrame(hist_l2_regularization_rows)
hist_complete_rows.extend(hist_l2_regularization_rows)
hist_l2_regularization_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_l2_regularization_winner = hist_l2_regularization_sweep.loc[hist_l2_regularization_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_l2_regularization_winner["configuration"], "train": hist_l2_regularization_winner["train"],
    "validation": hist_l2_regularization_winner["validation"], "model": hist_l2_regularization_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The strongest penalty lowers training F1 from 0.9769 to 0.9387 and leaves validation F1 at 0.8637, against 0.8626 without a penalty.**

__Step 57:__ Change only histogram boosting `max_bins` on the fixed training folds, with the sweep list `[63, 127, 255]`.

### `histogram boosting: max_bins`

Numeric values are grouped before fitting, and trees can split only between bins. The default is `255`, plus a separate bin for missing values. Fewer bins are expected to fit faster but may remove a useful threshold. The candidate list is `[63, 127, 255]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_max_bins_values = [63, 127, 255]
# One row per value will hold its scores, its seconds and its fitted model.
hist_max_bins_rows = []
for value in tqdm(hist_max_bins_values, desc="HistGradientBoostingClassifier.max_bins", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_bins=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_max_bins_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_max_bins_sweep = pd.DataFrame(hist_max_bins_rows)
hist_complete_rows.extend(hist_max_bins_rows)
hist_max_bins_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_bins_winner = hist_max_bins_sweep.loc[hist_max_bins_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_bins_winner["configuration"], "train": hist_max_bins_winner["train"],
    "validation": hist_max_bins_winner["validation"], "model": hist_max_bins_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Coarser bins score slightly higher: validation F1 is 0.8670 with 63 bins, 0.8636 with 127 and 0.8626 with 255.**

__Step 58:__ Plot training and validation F1 for every other swept histogram boosting setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
hist_settings_tables = {"learning_rate": (hist_learning_rate_sweep, hist_learning_rate_values), "max_leaf_nodes": (hist_max_leaf_nodes_sweep, hist_max_leaf_nodes_values), "l2_regularization": (hist_l2_regularization_sweep, hist_l2_regularization_values), "max_bins": (hist_max_bins_sweep, hist_max_bins_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), hist_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="F1", ylim=(0, 1.05))
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(hist_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**Across the four panels validation F1 stays between 0.8554 and 0.8670, while training F1 reaches 1.0000 at a learning rate of 0.3 and 0.9769 at the reference settings.** Training F1 is 0.8939 with 7 leaves per tree and 0.9387 with the strongest penalty, below the reference.

__Step 59:__ Draw the validation F1 of histogram boosting at every stage for four learning rates.

### `histogram boosting: learning rate and iteration count`

Smaller corrections often need more stages, but noise, overfitting and early termination can break that pattern. The first study plots actual fitted stages against validation score at each learning rate, with an explicit larger stage budget. The second enables a stopping criterion; validation determines whether its chosen count helps.

In [ ]:
# Four learning rates, each with its own staged validation curve (mean over the ten folds).
hist_interaction_reference = clone(hist_reference).set_params(early_stopping=False)
hist_curve_rates = [0.03, 0.1, 0.3, 1.0]
hist_rate_curves = {}
# Fit every fold once per rate at the largest stage count, then score the prediction after each stage: one fit gives the whole curve.
for rate in tqdm(hist_curve_rates, desc="HistGradientBoostingClassifier learning rates", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_interaction_reference).set_params(learning_rate=rate, max_iter=300).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    hist_rate_curves[rate] = np.mean([[f1_score(fold["y_score"], on_score) for on_score in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)], axis=0)
# Draw the curves; the dot marks the stage with the best validation score of each rate.
fig, ax = plt.subplots(figsize=(7, 5))
for rate, curve in hist_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter(np.argmax(curve) + 1, curve.max(), zorder=3)
ax.set(xlabel="max_iter", ylabel="validation F1", title="HistGradientBoostingClassifier: validation by stage at four learning rates")
ax.legend()
plt.tight_layout()
plt.show()
# Table of the same curves: the best stage, the stage where each curve first comes close to its best, and that best score.
pd.DataFrame({"best stage": [int(np.argmax(curve)) + 1 for curve in hist_rate_curves.values()],
    "stages to come within 1 percent of the best": [int(np.argmax(curve >= 0.99 * curve.max())) + 1 for curve in hist_rate_curves.values()],
    "best validation F1": [curve.max() for curve in hist_rate_curves.values()]}, index=pd.Index(hist_curve_rates, name="learning_rate")).round(4)

**A rate of 0.03 reaches validation F1 0.8675 at stage 159, a rate of 0.1 reaches 0.8688 at stage 21, a rate of 0.3 reaches 0.8645 at stage 8 and a rate of 1.0 peaks at 0.8511 at stage 157.** Within 1 percent of its best, the rate 0.03 needs 15 stages, the rate 0.1 needs 6, the rate 0.3 needs 2 and the rate 1.0 needs 1.

__Step 60:__ Stop each histogram boosting fit with its convergence criterion and count the stages it used.

In [ ]:
# With a convergence criterion each fit stops by itself, so the learning rate decides how many stages are fitted.
hist_stop_rows = []
for rate in tqdm(hist_curve_rates, desc="HistGradientBoostingClassifier early stopping", unit="rate"):
    started = time.perf_counter()
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).set_params(learning_rate=rate, max_iter=1000, early_stopping=True, n_iter_no_change=10, validation_fraction=0.1).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    hist_stop_rows.append({"learning_rate": rate, "stages fitted": np.mean([fitted.n_iter_ for fitted in fits]),
        "validation F1": np.mean([f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]),
        "seconds": time.perf_counter() - started})
pd.DataFrame(hist_stop_rows).set_index("learning_rate").round(4)

**With `early_stopping` on, the fit stops after 148.1 iterations on average over the folds at a rate of 0.03, 50.4 at 0.1, 24.9 at 0.3 and 11.8 at 1.0.** The validation F1 is 0.8637, 0.8653, 0.8620 and 0.8344, so the largest rate stops first and scores lowest.

__Step 61:__ Select the best tested histogram boosting configuration from the existing sweep rows by validation score.

In [ ]:
hist_complete = pd.DataFrame(hist_complete_rows)
hist_best = hist_complete.loc[hist_complete["cv_F1"].idxmax()]
hist_best_parameters = hist_best["parameters"]
hist_best_configuration = clone(hist_reference).set_params(**hist_best_parameters)
print("best tested histogram boosting configuration:", hist_best["configuration"])

**The best complete histogram boosting configuration measured uses 63 bins, with validation F1 0.8670.**

# <font color='#E8800A'>Selection of the boosting model</font> <a class="anchor" id="sel1"></a>
[Back to TOC](#toc)

The three boosting families are selected together by one search. Nested cross-validation comes first: it scores the whole search procedure on outer-fold rows that the search never saw, which checks that the grid is viable. The search on all the training rows then picks the boosting winner from that grid, and that model joins the combiners in the combining notebook.

The grid covers the three boosting families: every predictive dimension swept above and the tree settings listed in the code. The leaf-growth and regularisation controls in neither remain at their documented defaults. Execution controls, seeds and warm starts are fixed; histogram early stopping is disabled so stage counts have the same meaning in every fold. Domain constraints and native categorical handling are not introduced without a feature-specific course hypothesis.

The full space holds far more valid combinations than a class can fit, so the search evaluates 60 of them, drawn uniformly without replacement from the enumerated space with a seed of 33 before any score is seen. The same 60 feed the nested estimate and the search on all the training rows. Bergstra and Bengio (2012) show that 60 random draws reach the best 5 percent of a space with about 95 percent probability ([Random search for hyper-parameter optimization](https://jmlr.org/papers/v13/bergstra12a.html)). With all parameter values supplied as lists, scikit-learn's [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) uses `ParameterSampler`, which samples uniformly without replacement over the combinations indexed by `ParameterGrid`, so every valid combination has the same chance. `CachedSearchCV` runs that search with the preparation cache on, so no preparation is repeated.

The search space crosses, within each family, every value swept in that family's section: the value lists are read from the sweep cells above, not retyped. The search also draws the tree settings that a single decision tree takes, with the values listed in the code.

__Step 62:__ Declare the conditional grids of the boosting families, which fix the search space before any search score exists.

In [ ]:
# The search space of the first selection: for each boosting family one conditional Cartesian product of the value lists swept above, read from the sweep cells and declared before any search score exists. Each block names its own preparation step.
# The value lists of the tree settings. The sweeps above keep them at their reference values, so they enter the search here and nowhere else.
tree_values = {
    "gb": {"min_samples_leaf": [1, 5, 20, 50], "min_samples_split": [2, 10, 50], "max_features": [None, "sqrt", "log2"], "ccp_alpha": [0.0, 0.0001, 0.001, 0.01]},
    "hist": {"max_depth": [2, 3, 5, 10, None], "min_samples_leaf": [1, 5, 20, 50], "class_weight": [None, "balanced"]}}
boost_grid = [
    {"preprocessor": [clone(preparation)], "estimator": [clone(ada_reference)],
     "estimator__estimator": [value for value in ada_estimator_values if not isinstance(value, PreparedEstimator)],
     "estimator__n_estimators": ada_n_estimators_values,
     "estimator__learning_rate": ada_learning_rate_values},
    # AdaBoost: every base learner crossed with every stage count, learning rate and loss swept above.
    {"preprocessor": [AdaRawFrame()], "estimator": [clone(ada_reference)],
     "estimator__estimator": [value for value in ada_estimator_values if isinstance(value, PreparedEstimator)],
     "estimator__n_estimators": ada_n_estimators_values,
     "estimator__learning_rate": ada_learning_rate_values},
    # Gradient boosting: every combination of the settings swept in its section; a setting that only some losses use sits in its own block.
    {"preprocessor": [clone(preparation)], "estimator": [clone(gb_reference)],
     "estimator__n_estimators": gb_n_estimators_values,
     "estimator__learning_rate": gb_learning_rate_values,
     "estimator__max_depth": gb_max_depth_values,
     "estimator__min_samples_leaf": tree_values["gb"]["min_samples_leaf"],
     "estimator__min_samples_split": tree_values["gb"]["min_samples_split"],
     "estimator__subsample": gb_subsample_values,
     "estimator__max_features": tree_values["gb"]["max_features"],
     "estimator__loss": gb_loss_values,
     "estimator__ccp_alpha": tree_values["gb"]["ccp_alpha"]},
    # Histogram boosting: the same crossing for its settings.
    {"preprocessor": [clone(preparation)], "estimator": [clone(hist_reference)],
     "estimator__max_iter": hist_max_iter_values,
     "estimator__learning_rate": hist_learning_rate_values,
     "estimator__max_leaf_nodes": hist_max_leaf_nodes_values,
     "estimator__max_depth": tree_values["hist"]["max_depth"],
     "estimator__min_samples_leaf": tree_values["hist"]["min_samples_leaf"],
     "estimator__l2_regularization": hist_l2_regularization_values,
     "estimator__max_bins": hist_max_bins_values,
     "estimator__class_weight": tree_values["hist"]["class_weight"]}
]

__Step 63:__ Enumerate every valid combination, and draw 60 of them without replacement before any score is seen.

In [ ]:
# The conditional grids hold every valid combination; the search draws 60 of them, uniformly and without replacement, before any score exists.
print("combinations by group:", [len(ParameterGrid(group)) for group in boost_grid])
print("combinations in the space:", len(ParameterGrid(boost_grid)))

**The conditional grids of the boosting families hold 110310 valid combinations, and the search evaluates 60 of them.**

<div class="alert alert-block alert-info">
<code>CachedSearchCV(search)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
Wraps a scikit-learn search with a `PreparedEstimator`, caching each distinct preprocessing setting once per training-row set. The Week 5 class implemented its own grid search; this wrapper accepts any scikit-learn search and keeps its results.

**Main Parameters:**
- `search`: unfitted search, such as `GridSearchCV`, `RandomizedSearchCV` or `HalvingGridSearchCV`, with a `PreparedEstimator`.

**Main Attributes (after fit):**
- `cv_results_`, `best_params_`, `best_score_`, `best_index_`, `best_estimator_`: those of the search.
- `search_`: the fitted search.

**Main Methods:**
- `fit(X, y)`: run the cached search and keep its results.
- `predict(X)`, `predict_proba(X)`, `score(X, y)`: those of the best estimator, refitted on all rows.

**Differences from scikit-learn:**
[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) repeats preprocessing per candidate and fold. This wrapper supplies a temporary disk cache keyed by settings and rows. For grids and random searches with integer `random_state`, it prefits distinct settings per fold. For a search that picks candidates as it goes, such as `HalvingGridSearchCV`, a lock lets concurrent workers wait for one fit. Candidates, scores and winner remain those of the wrapped search; estimators without a `memory` parameter run unchanged.
</div>

__Step 64:__ Wrap the grid and the preprocessing in one search, with three inner folds.

In [ ]:
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
boost_template = CachedSearchCV(RandomizedSearchCV(PreparedEstimator(AdaRawFrame(), clone(ada_reference)), boost_grid, n_iter=60, random_state=random_state, cv=inner, scoring="f1", n_jobs=4))

__Step 65:__ Count the candidates and the model fits the search can cost.

In [ ]:
boost_count = boost_template.search.n_iter
print("inner-fold candidate fits per search:", boost_count * inner.get_n_splits())
print("outer splits:", folds.get_n_splits(), "inner splits:", inner.get_n_splits())
print("candidate fits plus winner refits:", (folds.get_n_splits() + 1) * (boost_count * inner.get_n_splits() + 1))

__Step 66:__ Estimate the whole search with the ten outer folds of the training partition, returning scores and the settings each outer search chose.

In [ ]:
boost_nested_started = time.perf_counter()
# F1 selects; ROC-AUC and balanced accuracy are companions scored on the same outer fits. A hard vote gives no probabilities, so a fold whose winner is a hard vote has no ROC-AUC and scores NaN there.
companions = {"f1": "f1", "balanced_accuracy": "balanced_accuracy",
    "roc_auc": lambda search, X, y: roc_auc_score(y, search.predict_proba(X)[:, 1]) if hasattr(search.best_estimator_.estimator_, "predict_proba") else np.nan}
# Each of the ten outer folds runs the whole search on its own training part, one fold per worker; the search inside uses four workers.
outer_runs = joblib.Parallel(n_jobs=10, return_as="generator")(
    joblib.delayed(cross_validate)(boost_template, X_train, y_train, cv=[split],
        scoring=companions, return_train_score=True, n_jobs=1, return_estimator=True, error_score="raise")
    for split in folds.split(X_train, y_train))
# Collect the outer results as the workers finish; the progress bar fills in bursts.
boost_fold_results = list(tqdm(outer_runs, total=folds.get_n_splits(),
                               desc="Nested CV, boosting: outer folds", unit="fold", leave=True, position=0))
# The outer validation scores estimate the whole procedure, selection step included; the training scores sit beside them, and ROC-AUC and balanced accuracy come from the same outer fits.
boost_nested_scores = {key: np.concatenate([result[key] for result in boost_fold_results]) for key in ("train_f1", "test_f1", "test_roc_auc", "test_balanced_accuracy")}
boost_nested_validation = boost_nested_scores["test_f1"]
boost_nested_train = boost_nested_scores["train_f1"]
# The settings each outer search chose are kept as plain settings; the fitted searches are dropped.
boost_fold_settings = [result["estimator"][0].best_params_ for result in boost_fold_results]
boost_nested_seconds = time.perf_counter() - boost_nested_started
# Report the estimate, its spread over the folds and the time.
print(f"nested validation F1: {boost_nested_validation.mean():.4f}")
print(f"fold SD: {boost_nested_validation.std(ddof=1):.4f}")
# ROC-AUC averages over the outer folds whose winner gives probabilities; balanced accuracy covers all of them.
print(f"ROC-AUC over the {np.isfinite(boost_nested_scores['test_roc_auc']).sum()} of {len(boost_nested_scores['test_roc_auc'])} outer folds whose winner gives probabilities")
print(f"nested wall time: {boost_nested_seconds:.2f} seconds")

**The complete search scores a nested validation F1 of 0.8620 with a fold SD of 0.0152.** The space holds 110310 valid combinations, 60 of them drawn for the search.

# <font color='#E8800A'>The boosting winner</font> <a class="anchor" id="boost_winner"></a>
[Back to TOC](#toc)

With the search space fixed and its nested estimate available, the same candidates are fitted to all the training rows. Its best estimator is the boosting winner: it supplies the predictions, the model artifact and one row of the course board, even when it differs from a default or from a sweep winner.

__Step 67:__ Run the declared grid of the boosting families on all the training rows and keep its winner.

In [ ]:
boost_started = time.perf_counter()
boost_search = clone(boost_template).fit(X_train, y_train)
boosting_winner = boost_search.best_estimator_
boost_seconds = time.perf_counter() - boost_started
print("chosen parameters:", boost_search.best_params_)
print("winner own inner-CV score:", boost_search.best_score_)
print("combinations evaluated:", len(boost_search.cv_results_["params"]))
print(f"search wall time: {boost_seconds:.2f} seconds")

__Step 68:__ Score the boosting winner on the ten outer folds, and log it beside its test score.

In [ ]:
# Score the winner on the same folds: its settings are fitted on each prepared training part. A winner whose base learner takes the raw-row path uses the folds of that path.
boost_folds = ada_prepared if isinstance(boosting_winner.preprocessor, AdaRawFrame) else prepared
boost_train, boost_validation = [], []
for fold in tqdm(boost_folds, desc='prepared', leave=True, position=0):
    fitted = clone(boosting_winner.estimator).fit(fold["X_fit"], fold["y_fit"])
    boost_train.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
    boost_validation.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))

In [ ]:
# The boosting winner as a plain constructor: the estimator of the winning block with the settings the search chose.
boost_settings = boost_search.best_params_
boosting_member = clone(boost_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in boost_settings.items() if key.startswith("estimator__")})
# Name the winner by the settings that differ from the defaults.
boost_label = " ".join(str(boosting_member).split())
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": boost_label, "train": boost_train, "validation": boost_validation, "model": boosting_winner},
    X_test, y_test, "winner's own score", metric=f1_score, label="F1")

__Step 69:__ Log the boosting winner with its nested estimate as the validation score.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": f"boosting search over {len(boost_search.cv_results_['params'])} combinations", "train": boost_nested_train, "validation": boost_nested_validation, "model": boosting_winner},
    X_test, y_test, "nested estimate", metric=f1_score, label="F1")

__Step 70:__ Read the test score of the boosting winner from the log.

In [ ]:
boost_test = model_log[-1]["test F1"]

**The search chose histogram boosting with 90 stages, at most 15 leaves, 5 rows per leaf and 63 bins, with held-out F1 0.8699 and an inner-CV score of 0.8672.** Its own score over the ten outer folds is 0.8649, against a nested estimate of 0.8620 for the whole search; the search chose the winner on the rows behind the first number, so the nested estimate is the one that describes the procedure.

__Step 71:__ Read every logged attempt before saving the result.

In [ ]:
print(pd.DataFrame(model_log).to_string(index=False))

# <font color='#E8800A'>The boosting result</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The nested estimate includes model selection. The fitted winner and each outer-training selection are saved for the combining notebook.

<div class="alert alert-block alert-info">
<code>save_boosting_selection(path, payload, model)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:** Saves the selected parameter dictionaries, fold scores and fitted winner.

**Main Parameters:** `path` names the selection file; `payload` contains task, context, folds, settings and scores; `model` is the fitted winner.

**Returns:** None.

**Differences from scikit-learn:** A search keeps settings and scores in memory. This course helper writes them with task and fold fingerprints, using [joblib](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html), so another notebook can validate and reuse them.
</div>

__Step 72:__ Save the boosting selection and fitted winner with their data and fold fingerprints.

In [ ]:
# Fingerprint the data, recipe, split and exact outer rows.
selection_context = result_context(
    "../../data/interim/champions.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train))
fold_fingerprint = joblib.hash(outer_folds(folds, X_train, y_train))
boosting_dir = Path(results_dir) / "boosting"
boosting_dir.mkdir(parents=True, exist_ok=True)
# Compare fixed settings with the same family's default on the same ten folds.
boost_family = type(boosting_winner.estimator).__name__
boost_reference = next(row for row in model_log
    if row.get("validation F1 is") == "untuned" and row["model"].startswith(boost_family + "("))
boost_baseline_cv = float(boost_reference["validation F1"])
boost_baseline_test = float(boost_reference["test F1"])
boost_tuning_comparison = dict(
    protocol="fixed_settings_on_course_folds", family=boost_family,
    candidate_model=boost_label, reference_model=boost_reference["model"],
    reference_kind="library_default", candidate_cv=float(np.mean(boost_validation)),
    reference_cv=boost_baseline_cv, candidate_test=float(boost_test), reference_test=boost_baseline_test)
# Keep only the selected settings and the scores the combining part needs.
selection_payload = {
    "task": "classification", "context": selection_context, "fold_fingerprint": fold_fingerprint,
    "boost_settings": boost_settings, "boost_fold_settings": boost_fold_settings,
    "boost_nested_scores": boost_nested_scores,
    "boost_nested_validation": boost_nested_validation, "boost_nested_train": boost_nested_train,
    "boost_train": boost_train, "boost_validation": boost_validation,
    "boost_label": boost_label, "boost_test": float(boost_test),
    "baseline_cv": boost_baseline_cv, "baseline_test": boost_baseline_test,
    "tuning_comparison": boost_tuning_comparison,
}
save_boosting_selection(boosting_dir / "week_11_classification_selection.joblib", selection_payload, boosting_winner)
print("Saved boosting selection:", boosting_dir / "week_11_classification_selection.joblib")

__Step 73:__ Save and read the boosting result.

In [ ]:
board = Scoreboard("champions Outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train)),
    floor=f1_score(y_test, DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)))
for week, earlier_model, earlier_cv, earlier_test in results_before(11, "classification", results_dir, metric=board.metric, context=board.context):
    board.record(week, earlier_model, earlier_cv, earlier_test)
board.record(11, boost_label, round(float(np.mean(boost_nested_validation)), 4), round(float(boost_test), 4))
board.save(boosting_dir, 11, "classification", baseline_cv=round(boost_baseline_cv, 4), baseline_test=round(boost_baseline_test, 4))
# Keep the fixed-setting comparison separate from the nested procedure estimate.
boost_record_path = boosting_dir / "week_11.json"
boost_record = json.loads(boost_record_path.read_text(encoding="utf-8"))
boost_record["classification"]["tuning_comparison"] = boost_tuning_comparison
boost_record_path.write_text(json.dumps(boost_record, indent=1) + "\n", encoding="utf-8")
print(board.table())
# Companion metrics use the same outer fits and final winner, without further training.
companion_metrics = pd.DataFrame({
    "nested outer-fold mean": {"F1": boost_nested_validation.mean(), "ROC-AUC": np.nanmean(boost_nested_scores["test_roc_auc"]), "balanced accuracy": boost_nested_scores["test_balanced_accuracy"].mean()},
    "held-out winner": {"F1": boost_test, "ROC-AUC": roc_auc_score(y_test, boosting_winner.predict_proba(X_test)[:, 1]), "balanced accuracy": balanced_accuracy_score(y_test, boosting_winner.predict(X_test))}}).round(4)
display(companion_metrics)

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. AdaBoost starts from its library defaults; each isolated sweep changes only its named setting.
2. AdaBoost reweights rows; gradient boosting fits trees to loss gradients.
3. Smaller corrections often need more stages, but validation determines the useful stopping point.
4. A nested estimate scores the search procedure, including its selection step.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

Gradient boosting reaches its best plotted validation F1 of 0.8641 after 291 stages at rate 0.03, and 0.8643 after 121 stages at rate 0.1. Why can a smaller learning rate need more stages?

**Answer.** A smaller learning rate reduces each stage's influence, so more stages may be needed to reach a similar fit. Validation still determines whether those extra stages help.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

Original papers support the method explanations; library documentation defines the estimator contracts.

- [scikit-learn ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)
- [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)
- [HistGradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html)
- [Freund, Y. and Schapire, R. E. (1997). A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting. Journal of Computer and System Sciences, 55(1), 119-139.](https://doi.org/10.1006/jcss.1997.1504)
- [Friedman, J. H., Hastie, T. and Tibshirani, R. (2000). Additive Logistic Regression: A Statistical View of Boosting. Annals of Statistics, 28(2), 337-407.](https://doi.org/10.1214/aos/1016218223)
- [Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. Annals of Statistics, 29(5), 1189-1232.](https://doi.org/10.1214/aos/1013203451)
- [Zhu, J., Zou, H., Rosset, S. and Hastie, T. (2009). Multi-class AdaBoost. Statistics and Its Interface, 2(3), 349-360.](https://doi.org/10.4310/SII.2009.v2.n3.a8)
- [Bergstra, J. and Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. Journal of Machine Learning Research, 13(10), 281-305.](https://jmlr.org/papers/v13/bergstra12a.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [joblib disk-based caching](https://joblib.readthedocs.io/en/latest/user_guide/memory.html)
- [joblib serialization and loading](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html)